# Goodreads Machine Learning - Group Project
------------
------------


## Table of contents

1.  [Data loading and first look](#1.-Data-loading-and-first-look)
    - [Load source files](#Load-source-files)
    - [First look](#First-look)
2.  [Cleaning and exploratory data analysis (EDA)](#2.-Cleaning-and-exploratory-data-analysis-%28EDA%29)
    - [2.1 General cleaning](#2.1-General-cleaning)
    - [2.2 `publication_date`](#2.2-publication_date)
    - [2.3 `average_rating`](#2.3-average_rating)
    - [2.4 `ratings_count`](#2.4-ratings_count)
    - [2.5 `num_pages`](#2.5-num_pages)
    - [2.6 `text_reviews_count`](#2.6-text_reviews_count)
    - [2.7 `language_code`](#2.7-language_code)
    - [2.8 `authors`](#2.8-authors)
    - [2.9 `title`](#2.9-title)
    - [2.10 `publisher`](#2.10-publisher)
3.  [Modelling](#3.-Modelling)
    - [3.1 Modelling setup](#3.1-Modelling-setup)
    - [3.2 Modelling source table](#3.2-Modelling-source-table)
    - [3.3 Train/test split](#3.3-Train/test-split)
    - [3.4 Feature Description](#3.4-Feature-Description)
    - [3.5 Feature Engineering](#3.5-Feature-Engineering)
    - [3.6 Feature Selection](#3.6-Feature-Selection)
    - [3.7 Active Model Training](#3.7-Active-Model-Training)
    - [3.8 Model tuning](#3.8-Model-tuning)
    - [3.9 Final evaluation](#3.9-Final-evaluation)
    - [3.10 Feature Impact Analysis](#3.10-Feature-Impact-Analysis)
    - [3.11 Hold-out prediction diagnostics](#3.11-Hold-out-prediction-diagnostics)
4.  [Extended Goodreads dataset](#4.-Extended-Goodreads-dataset)
    - [Extended-data imports and export controls](#Extended-data-imports-and-export-controls)
    - [Validate the saved cleaned dataset](#Validate-the-saved-cleaned-dataset)
    - [Apply deterministic extended-data cleaning](#Apply-deterministic-extended-data-cleaning)
    - [Resolve true duplicates within repeated ISBN groups](#Resolve-true-duplicates-within-repeated-ISBN-groups)
    - [Load and clean the raw source once](#Load-and-clean-the-raw-source-once)
    - [Export the cleaned dataset safely](#Export-the-cleaned-dataset-safely)
    - [Run the guarded cleaning export](#Run-the-guarded-cleaning-export)
    - [Load the saved cleaned dataset](#Load-the-saved-cleaned-dataset)
    - [Review the stored cleaning audit](#Review-the-stored-cleaning-audit)
    - [Compare the original and extended modelling populations](#Compare-the-original-and-extended-modelling-populations)
    - [Compare cleaned distributions visually](#Compare-cleaned-distributions-visually)
    - [Create the extended train/test split](#Create-the-extended-train/test-split)
    - [Check extended rating-band balance](#Check-extended-rating-band-balance)
    - [Fit extended feature engineering on training rows](#Fit-extended-feature-engineering-on-training-rows)
    - [Check overlap with the original holdout](#Check-overlap-with-the-original-holdout)
    - [Evaluate the selected XGBoost configuration across datasets](#Evaluate-the-selected-XGBoost-configuration-across-datasets)
5.  [Model export](#5.-Model-export)
    - [5.1 Export controls](#5.1-Export-controls)
    - [5.2 Exportable predictor](#5.2-Exportable-predictor)
    - [5.3 Refit and export](#5.3-Refit-and-export)
    - [5.4 Reload verification](#5.4-Reload-verification)

------------
## 1. Data loading and first look

Loading the original CSV to document its parsing problem, then loading the repaired source data and checking its structure before cleaning.

#### Load project libraries

In [ ]:
import re
import unicodedata
import warnings
from typing import Any

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
import seaborn as sns
import plotly.express as px


pd.set_option("display.max_colwidth", None)

sns.set_theme(
    context="notebook",
    style="whitegrid",
    rc={
        "figure.figsize": (10, 5.5),
        "figure.dpi": 110,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#D7DEE5",
        "grid.alpha": 0.55,
    },
)

COLOR_PRIMARY = "#167D77"
COLOR_ACCENT = "#D97706"
COLOR_BLUE = "#35618D"
COLOR_CORAL = "#C8553D"
COLOR_GOLD = "#E3B341"
COLOR_NEUTRAL = "#6B7280"
COLOR_LIGHT = "#D9E7E5"
PLOT_RANDOM_STATE = 42

DATASET_PALETTE = {
    "Original": COLOR_BLUE,
    "Extended cleaned": COLOR_PRIMARY,
}

### Load source files


In [ ]:
CSV_RAW_PATH = "../data/raw/books.csv"

with warnings.catch_warnings(record=True) as parser_warnings:
    warnings.simplefilter("always", pd.errors.ParserWarning)
    df_raw_parse_check = pd.read_csv(
        CSV_RAW_PATH,
        index_col="bookID",
        on_bad_lines="warn",
    )

skipped_raw_rows = sum(
    str(parser_warning.message).count("Skipping line")
    for parser_warning in parser_warnings
)
print(f"Rows skipped because they contain too many CSV fields: {skipped_raw_rows}")

The parser identifies four malformed rows. Each contains an author name with a comma that was treated as an extra CSV separator, producing 13 fields instead of the expected 12. The repaired file removes those separators from the affected author values.

#### Load the repaired dataset

In [ ]:
CSV_REPAIRED_PATH = "../data/processed/books-repaired.csv"
df_raw = pd.read_csv(CSV_REPAIRED_PATH, on_bad_lines="warn")


Three dataframes are kept through the notebook:

- `df_raw` contains the unchanged repaired source data;
- `df_clean` contains documented cleaning changes;
- `df_model` contains exploratory or modelling features and row filters.

### First look


In [ ]:
df_raw.head()


#### Inspect column types and completeness

In [ ]:
df_raw.info()


The first inspection shows surrounding spaces in the `num_pages` column name. `publication_date` is also stored as text and must be converted before date-based analysis.

------------
## 2. Cleaning and exploratory data analysis (EDA)

Creating separate cleaned and exploratory dataframes, documenting each data rule, and examining how the available book information relates to `average_rating`.

### 2.1 General cleaning


#### Initial observations

- The `num_pages` column name contains leading spaces.
- `isbn13` is numeric even though it is an identifier rather than a measured quantity.
- `publication_date` is text and must be parsed as a date.

#### Check identifier uniqueness

Confirming whether the book identifiers are unique before choosing one of them as the dataframe index.


In [ ]:
# check if bookID, ISBN, ISBN13 are all unique
identifier_columns = ["bookID", "isbn", "isbn13"]
{column_name: df_raw[column_name].is_unique for column_name in identifier_columns}


#### Create the cleaning dataframe

Copying the source data before setting `bookID` as the cleaning index keeps `df_raw` unchanged while allowing precise row-level corrections.

In [ ]:
df_clean = df_raw.copy()

#### Trim surrounding spaces from column names

Removing accidental spaces from the cleaning copy makes column references consistent.

In [ ]:
# Remove surrounding spaces from column names in the cleaning copy.
df_clean.columns = df_clean.columns.str.strip()

#### Check string whitespace

Scanning text columns for leading or trailing spaces before copying the data into the cleaning dataframe.


In [ ]:
# Check whether text values contain leading or trailing spaces.
string_columns = df_clean.select_dtypes(include=["object", "string"]).columns
for column_name in string_columns:
    non_missing_values = df_clean[column_name].dropna()
    has_leading_or_trailing_spaces = (
        non_missing_values.ne(non_missing_values.str.strip()).any()
    )
    if has_leading_or_trailing_spaces:
        print(f"Leading/trailing spaces detected in column: {column_name}")
    else:
        print(f"No surrounding spaces detected in column: {column_name}")

#### Set the cleaning index and validate the stages

Moving `bookID` to the index only in `df_clean` preserves the source table and makes later row corrections explicit.

In [ ]:
df_clean = df_clean.set_index("bookID")

cleaning_stage_summary = pd.DataFrame([
    {
        "dataframe": "df_raw",
        "rows": len(df_raw),
        "data_columns": df_raw.shape[1],
        "bookID_location": "column",
    },
    {
        "dataframe": "df_clean",
        "rows": len(df_clean),
        "data_columns": df_clean.shape[1],
        "bookID_location": "index",
    },
])
cleaning_stage_summary

#### Trim title values

Removing surrounding spaces from titles before broader text normalization.

In [ ]:
# fix spaces in titles
df_clean["title"] = df_clean["title"].str.strip()


#### Normalize strings

In [ ]:
def normalize_string(
    value: Any,
    *,
    lowercase: bool = True,
    strip_accents: bool = True,
    collapse_whitespace: bool = True,
    remove_suffixes: bool = False,
    remove_punctuation: bool = False,
) -> str | None:
    """
    Normalize one string for consistent matching and grouping.
    Returns None for None/NaN. Non-strings are cast to str first.
    """
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    normalized_text = str(value).strip()
    if not normalized_text:
        return ""
    if lowercase:
        normalized_text = normalized_text.lower()
    if strip_accents:
        normalized_text = unicodedata.normalize("NFKD", normalized_text)
        normalized_text = "".join(
            character for character in normalized_text if not unicodedata.combining(character)
        )
    if collapse_whitespace:
        normalized_text = re.sub(r"\s+", " ", normalized_text)
    if remove_suffixes:
        normalized_text = re.sub(
            r"\s+(jr\.?|sr\.?|iii|ii|iv)$",
            "",
            normalized_text,
            flags=re.IGNORECASE,
        ).strip()
    if remove_punctuation:
        normalized_text = re.sub(r"[^\w\s]", "", normalized_text)
        normalized_text = re.sub(r"\s+", " ", normalized_text).strip()
    return normalized_text


#### Normalize main text fields

In [ ]:
df_clean["title"] = df_clean["title"].apply(normalize_string)
df_clean["authors"] = df_clean["authors"].apply(normalize_string)
df_clean["publisher"] = df_clean["publisher"].apply(normalize_string)


#### Blank, empty, null, and bad data


In [ ]:
df_clean.describe()

The summary shows zero values in `num_pages` and `text_reviews_count`. These values require separate investigation because zero can be valid for review counts but usually represents missing page information.

#### Check explicit missing values

In [ ]:
df_clean.isna().sum()


#### Check null-like text values

In [ ]:
# check for bad data
null_like_values = ["", "nan", "None", "N/A", "null", "-"]
null_like_value_counts = {}
for column_name in df_clean.columns:
    null_like_mask = df_clean[column_name].astype(str).str.strip().str.lower().isin(null_like_values)
    null_like_value_counts[column_name] = int(null_like_mask.sum())
null_like_value_counts


No standard missing values are present at this stage, but five records use `NOT A BOOK` as the author label and require review.

#### Count invalid author labels

In [ ]:
print("number of books with \'NOT A BOOK\' as authors :", (df_clean["authors"] == "NOT A BOOK".lower()).sum())

#### Inspect invalid author records

In [ ]:
df_clean.loc[df_clean["authors"] == "NOT A BOOK".lower(),:]

The records are audio productions or podcasts rather than books with a valid author value.

**Decision:** Remove these rows because the author field is invalid for the planned book-level model.

#### Drop invalid author rows

In [ ]:
invalid_author_mask = df_clean["authors"].eq("not a book")
invalid_author_rows_removed = int(invalid_author_mask.sum())
df_clean = df_clean.loc[~invalid_author_mask].copy()

print(f"Invalid author rows removed: {invalid_author_rows_removed}")
print(f"Invalid author rows remaining: {df_clean['authors'].eq('not a book').sum()}")

#### Zero values


In [ ]:
print("num_pages zero count :", (df_clean["num_pages"] == 0).sum())
print("ratings_count zero count :", (df_clean["ratings_count"] == 0).sum())
print("text_reviews_count zero count :", (df_clean["text_reviews_count"] == 0).sum())

These zero values are investigated in the following feature-specific sections before any cleaning decision is applied.

### 2.2 `publication_date`


#### Convert publication date to datetime


In [ ]:
# fix publication_date, passing format to parse
raw_publication_date = df_clean["publication_date"].copy()
df_clean["publication_date"] = pd.to_datetime(
    df_clean["publication_date"],
    format="%m/%d/%Y",
    errors="coerce", # if the format is not correct, set to NaT
)
# check original values where the conversion failed
raw_publication_date.loc[df_clean["publication_date"].isna()]


Two dates cannot be parsed because their day values are impossible. The edition identifiers make a targeted external check practical.

#### Inspect invalid publication dates

In [ ]:
df_clean.loc[df_clean.publication_date.isna(), ["isbn", "isbn13", "title"]]

Goodreads' edition listings report:

- *In Pursuit of the Proper Sinner* (ISBN `0553575104`) as published on [October 31, 2000](https://www.goodreads.com/work/editions/2604448-in-pursuit-of-the-proper-sinner-inspector-lynley-10);
- *Montaillou, village occitan de 1294 à 1324* (ISBN `2070323285`) as published on [June 30, 1982](https://www.goodreads.com/work/editions/44742-montaillou-village-occitant-de-1294-1324).

These are edition-specific corrections. Other editions of the same titles can have different dates.

**Decision:** Correct the two impossible dates using the matching edition records and retain both rows.

#### Correct and validate the two dates

Applying the edition-specific corrections and checking that no invalid dates remain.

In [ ]:
# Correct the two edition dates identified above.
df_clean.loc[31373, "publication_date"] = pd.to_datetime("2000-10-31")
df_clean.loc[45531, "publication_date"] = pd.to_datetime("1982-06-30")

print(f"Invalid publication dates remaining: {df_clean['publication_date'].isna().sum()}")

#### Initialize the exploratory dataframe and publication year

Starting `df_model` from the cleaned data and extracting the publication year for easier exploratory comparisons.

In [ ]:
df_model = df_clean.copy()
df_model["publication_year"] = df_model["publication_date"].dt.year

#### Visualize edition dates by decade

Book counts show the dataset's time coverage. A logarithmic count scale keeps the sparsely represented early decades visible without hiding the concentration in recent decades.

In [ ]:
publication_decade_summary = (
    df_model.assign(
        publication_decade=(df_model["publication_year"] // 10 * 10).astype(int)
    )
    .groupby("publication_decade")
    .agg(books=("average_rating", "size"))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 5.3))
decade_bars = ax.bar(
    publication_decade_summary["publication_decade"].astype(str),
    publication_decade_summary["books"],
    color=COLOR_PRIMARY,
    alpha=0.88,
    edgecolor="white",
    linewidth=0.8,
)
ax.set_yscale("log")
ax.bar_label(
    decade_bars,
    labels=publication_decade_summary["books"],
    padding=3,
    fontsize=9,
)
ax.set(
    title="Edition dates are concentrated in the 1990s and 2000s",
    xlabel="Publication decade",
    ylabel="Number of books (log scale)",
)
ax.grid(axis="x", visible=False)

plt.tight_layout()
plt.show()

About 90% of the cleaned records are editions published in the 1990s or 2000s. Earlier decades contain far fewer books, so comparisons involving those decades would be unstable.

### 2.3 `average_rating`


#### Plot average rating distribution

In [ ]:
average_ratings = df_model["average_rating"].dropna()
mean_rating = average_ratings.mean()
median_rating = average_ratings.median()
rating_05, rating_95 = average_ratings.quantile([0.05, 0.95])

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.axvspan(
    rating_05,
    rating_95,
    color=COLOR_PRIMARY,
    alpha=0.08,
    label=f"Middle 90%: {rating_05:.2f}–{rating_95:.2f}",
)
sns.histplot(
    average_ratings,
    binwidth=0.10,
    stat="percent",
    kde=True,
    color=COLOR_PRIMARY,
    alpha=0.78,
    line_kws={"linewidth": 2.2},
    ax=ax,
)
ax.axvline(
    mean_rating,
    color=COLOR_CORAL,
    linestyle="--",
    linewidth=2,
    label=f"Mean: {mean_rating:.3f}",
)
ax.axvline(
    median_rating,
    color=COLOR_BLUE,
    linestyle=":",
    linewidth=2.4,
    label=f"Median: {median_rating:.2f}",
)
ax.set(
    xlim=(max(0, average_ratings.min() - 0.1), 5),
    title="Goodreads ratings cluster tightly around four stars",
    xlabel="Average rating (0–5)",
    ylabel="Share of books",
)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
ax.legend(frameon=False, ncol=3, loc="upper left")
ax.grid(axis="x", visible=False)

plt.tight_layout()
plt.show()

average_rating_distribution_metrics = pd.DataFrame({
    "metric": [
        "mean",
        "median",
        "standard_deviation",
        "5th_percentile",
        "95th_percentile",
        "minimum",
        "maximum",
        "count",
    ],
    "value": [
        mean_rating,
        median_rating,
        average_ratings.std(),
        rating_05,
        rating_95,
        average_ratings.min(),
        average_ratings.max(),
        average_ratings.count(),
    ],
})

average_rating_distribution_metrics

#### Interpretation

The distribution is concentrated around a mean rating of approximately 3.93. A small number of zero ratings require comparison with `ratings_count` before interpretation.

#### Check impossible ratings

Confirming that negative average ratings are absent before creating rating groups.

In [ ]:
print("average_rating negative count :",(df_model["average_rating"] < 0).sum())


#### Create average-rating deciles

Dividing the target into ten approximately equal-sized groups makes the low, middle, and high rating ranges easier to compare.

In [ ]:
average_rating_decile_labels = [
    "N1", "N2", "N3", "N4", "N5", "N6", "N7", "N8", "N9", "N10"
]
df_model["average_rating_decile"] = pd.qcut(
    df_model["average_rating"],
    q=10,
    labels=average_rating_decile_labels,
)
df_model.groupby("average_rating_decile")["average_rating"].describe()


The lowest and highest rating groups have the widest spread. This describes the target distribution but does not establish a cause.

### 2.4 `ratings_count`


#### First Look


In [ ]:
df_model["ratings_count"].describe()


#### Relation with `average_rating`


In [ ]:
# Plot a readable sample before summarizing the full relationship.
ratings_count_plot_sample = df_model.sample(
    n=min(3_000, len(df_model)),
    random_state=PLOT_RANDOM_STATE,
)

fig, ax = plt.subplots(figsize=(10, 5.3))
ax.scatter(
    ratings_count_plot_sample["ratings_count"],
    ratings_count_plot_sample["average_rating"],
    s=18,
    alpha=0.22,
    linewidth=0,
    color=COLOR_PRIMARY,
)
ax.set_xscale("symlog", linthresh=10)
ax.set_xlim(0, df_model["ratings_count"].max() * 1.05)
ratings_count_ticks = [0, 10, 100, 1_000, 10_000, 100_000, 1_000_000]
ax.set_xticks([
    tick for tick in ratings_count_ticks
    if tick <= df_model["ratings_count"].max()
])
ax.xaxis.set_major_formatter(FuncFormatter(
    lambda value, position: f"{value:,.0f}"
))
ax.axhline(
    df_model["average_rating"].median(),
    color=COLOR_CORAL,
    linestyle="--",
    linewidth=1.8,
    label="Overall median rating",
)
ax.set(
    title="Low-support books show the widest rating spread",
    xlabel="Number of ratings (symmetric log scale; 3,000-book sample)",
    ylabel="Average rating (0–5)",
    ylim=(0, 5),
)
ax.legend(frameon=False)
ax.grid(axis="x", visible=False)

plt.tight_layout()
plt.show()

#### Use a log rating-count scale for readability

A logarithmic scale compresses very large counts so books with fewer ratings remain visible without changing their order.

In [ ]:
# Keep only rows with a positive ratings count.
ratings_count_correlation_df = df_model.loc[
    df_model["ratings_count"] > 0,
    ["ratings_count", "average_rating"],
].copy()

ratings_count_plot_df = ratings_count_correlation_df.copy()
ratings_count_plot_df["log_ratings_count"] = np.log10(
    ratings_count_plot_df["ratings_count"]
)
ratings_count_plot_df["support_quantile"] = pd.qcut(
    ratings_count_plot_df["log_ratings_count"],
    q=15,
    duplicates="drop",
)
ratings_count_trend = (
    ratings_count_plot_df.groupby("support_quantile", observed=True)
    .agg(
        log_ratings_count=("log_ratings_count", "median"),
        median_rating=("average_rating", "median"),
        rating_10=("average_rating", lambda values: values.quantile(0.10)),
        rating_90=("average_rating", lambda values: values.quantile(0.90)),
    )
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(10, 5.4))
ax.fill_between(
    ratings_count_trend["log_ratings_count"],
    ratings_count_trend["rating_10"],
    ratings_count_trend["rating_90"],
    color=COLOR_PRIMARY,
    alpha=0.16,
    label="Middle 80% of ratings",
)
ax.plot(
    ratings_count_trend["log_ratings_count"],
    ratings_count_trend["median_rating"],
    color=COLOR_PRIMARY,
    linewidth=2.5,
    marker="o",
    markersize=5,
    label="Median rating",
)
ax.axhline(
    ratings_count_plot_df["average_rating"].median(),
    color=COLOR_CORAL,
    linestyle="--",
    linewidth=1.7,
    label="Overall median",
)
largest_support_tick = int(np.floor(
    ratings_count_trend["log_ratings_count"].max()
))
rating_support_ticks = np.arange(0, largest_support_tick + 1)
rating_support_labels = ["1", "10", "100", "1k", "10k", "100k", "1m"]
ax.set_xticks(rating_support_ticks)
ax.set_xticklabels(rating_support_labels[:len(rating_support_ticks)])
ax.set(
    title="Typical ratings change little as reader support grows",
    xlabel="Number of ratings (log scale)",
    ylabel="Average rating (0–5)",
)
ax.legend(frameon=False, ncol=3, loc="lower right")
ax.grid(axis="x", visible=False)

plt.tight_layout()
plt.show()

The extreme 0–2 and 5-star averages occur only among books with fewer than ten ratings. The overall relationship remains weak, and positive averages paired with zero supporting ratings are internally inconsistent records that require investigation.

#### Inspect rating-count anomalies

Separating positive ratings with no supporting votes from rows where both values are zero.

In [ ]:
# number of books with ratings_count = 0 and average_rating > 0
print(f"Number of books with ratings_count = 0 and average_rating > 0: {len(df_model[(df_model['ratings_count'] == 0) & (df_model['average_rating'] > 0)])}")
# number of books with ratings_count = 0 and average_rating = 0
print(f"Number of books with ratings_count = 0 and average_rating = 0: {len(df_model[(df_model['ratings_count'] == 0) & (df_model['average_rating'] == 0)])}")


#### Build rating-count bands

Grouping books by rating support allows the mean rating and its spread to be compared across ranges.

In [ ]:
EDA_MIN_RATINGS_COUNT = 10
ratings_count_bins = [-1, 0, EDA_MIN_RATINGS_COUNT - 1, 25, 100, 1_000, 10_000, np.inf]
ratings_count_labels = ["0", "1-9", "10-25",
                        "26-100", "101-1k", "1k-10k", "10k+"]

df_model["ratings_count_band"] = pd.cut(
    df_model["ratings_count"],
    bins=ratings_count_bins,
    labels=ratings_count_labels,
    include_lowest=True,
)

ratings_count_summary = (
    df_model.groupby("ratings_count_band", observed=True)
    .agg(
        books=("average_rating", "size"),
        mean_rating=("average_rating", "mean"),
        median_rating=("average_rating", "median"),
        rating_std=("average_rating", "std"),
        median_ratings_count=("ratings_count", "median"),
    )
    .reset_index()
)
ratings_count_summary["share_of_df_model"] = ratings_count_summary["books"] / len(df_model)

ratings_count_summary

#### Plot rating-count band summary

In [ ]:
# Compare the number of books in each rating-count band with the rating spread inside each band.
ratings_count_plot_summary = ratings_count_summary.copy()
ratings_count_plot_summary["share_percent"] = (
    ratings_count_plot_summary["share_of_df_model"] * 100
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)
y_positions = np.arange(len(ratings_count_plot_summary))

support_bars = axes[0].barh(
    y_positions,
    ratings_count_plot_summary["share_percent"],
    color=COLOR_PRIMARY,
    alpha=0.88,
)
axes[0].bar_label(
    support_bars,
    labels=[f"{share:.1f}%" for share in ratings_count_plot_summary["share_percent"]],
    padding=4,
    fontsize=9,
)
axes[0].set(
    title="Where the books are",
    xlabel="Share of books",
    ylabel="Ratings-count band",
    yticks=y_positions,
    yticklabels=ratings_count_plot_summary["ratings_count_band"],
)
axes[0].xaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
axes[0].grid(axis="y", visible=False)

axes[1].hlines(
    y_positions,
    0,
    ratings_count_plot_summary["rating_std"],
    color=COLOR_LIGHT,
    linewidth=5,
)
axes[1].scatter(
    ratings_count_plot_summary["rating_std"],
    y_positions,
    s=85,
    color=COLOR_CORAL,
    zorder=3,
)
for y_position, rating_std in zip(
    y_positions,
    ratings_count_plot_summary["rating_std"],
):
    axes[1].text(rating_std + 0.025, y_position, f"{rating_std:.2f}", va="center")
axes[1].set(
    title="How variable their ratings are",
    xlabel="Standard deviation of average rating",
    yticks=y_positions,
    yticklabels=ratings_count_plot_summary["ratings_count_band"],
)
axes[1].grid(axis="y", visible=False)

fig.suptitle("Low-support books are rare but much less stable", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

Mean ratings are similar across most support bands. The 10,000-plus band is slightly higher, but the difference is small; rating spread is the clearer distinction.

**Decision:** Exclude books with fewer than ten ratings from later EDA because averages based on very few readers are visibly less stable. This exploratory rule does not imply that every removed row is corrupted.

#### Filter low-support rows

##### Apply the exploratory cutoff

The ten-rating threshold is used for later EDA. Final model training applies a separate, more conservative minimum of 50 ratings, declared in Section 3.2.

In [ ]:
# Keep rows meeting the exploratory rating-support cutoff.
df_model = df_model[
    df_model["ratings_count"] >= EDA_MIN_RATINGS_COUNT
].copy()

#### Check rows kept for modelling

In [ ]:
print('rows in df_clean :', len(df_clean))
print('rows in df_model :', len(df_model))
#numer of removed rows
print('number of removed rows :', len(df_clean) - len(df_model))

The filter removes 670 rows, approximately 6% of the cleaned data, reducing the influence of averages based on very few observations during EDA.

#### Recheck rating-count relationship

In [ ]:
# Replot the relationship after removing low-support rows.
filtered_support_plot_df = df_model[["ratings_count", "average_rating"]].copy()
filtered_support_plot_df["support_quantile"] = pd.qcut(
    filtered_support_plot_df["ratings_count"],
    q=12,
    duplicates="drop",
)
filtered_support_trend = (
    filtered_support_plot_df.groupby("support_quantile", observed=True)
    .agg(
        ratings_count=("ratings_count", "median"),
        median_rating=("average_rating", "median"),
        rating_10=("average_rating", lambda values: values.quantile(0.10)),
        rating_90=("average_rating", lambda values: values.quantile(0.90)),
    )
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.fill_between(
    filtered_support_trend["ratings_count"],
    filtered_support_trend["rating_10"],
    filtered_support_trend["rating_90"],
    color=COLOR_PRIMARY,
    alpha=0.18,
    label="Middle 80% of ratings",
)
ax.plot(
    filtered_support_trend["ratings_count"],
    filtered_support_trend["median_rating"],
    color=COLOR_PRIMARY,
    marker="o",
    linewidth=2.5,
    label="Median rating",
)
ax.set_xscale("log")
ax.set(
    title="After filtering, typical ratings remain stable as support grows",
    xlabel="Median ratings count within each equal-frequency group (log scale)",
    ylabel="Average rating (0–5)",
)
ax.legend(frameon=False)
ax.grid(axis="x", visible=False)

plt.tight_layout()
plt.show()

### 2.5 `num_pages`


#### First look

In [ ]:
# describe num_pages
df_model["num_pages"].describe()


#### Plot num_pages distribution

In [ ]:
# Plot the distribution of page counts to show where most books are concentrated.
page_count_plot_limit = df_model["num_pages"].quantile(0.995)
page_count_plot_data = df_model.loc[
    df_model["num_pages"] <= page_count_plot_limit,
    "num_pages",
]
median_page_count = df_model["num_pages"].median()

fig, ax = plt.subplots(figsize=(10, 5.5))
sns.histplot(
    page_count_plot_data,
    binwidth=50,
    stat="percent",
    color=COLOR_BLUE,
    alpha=0.82,
    edgecolor="white",
    ax=ax,
)
ax.axvline(
    median_page_count,
    color=COLOR_CORAL,
    linestyle="--",
    linewidth=2,
    label=f"Median: {median_page_count:.0f} pages",
)
ax.set(
    title="Most editions contain a few hundred pages",
    xlabel=f"Cleaned page count (shown through the 99.5th percentile: {page_count_plot_limit:.0f})",
    ylabel="Share of books",
)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
ax.legend(frameon=False)
ax.grid(axis="x", visible=False)

plt.tight_layout()
plt.show()

#### Count unusual number of pages


In [ ]:
LOW_PAGE_COUNT_THRESHOLD = 10
HIGH_PAGE_COUNT_THRESHOLD = 2000

print(
    f'books with less than {LOW_PAGE_COUNT_THRESHOLD} pages : {(df_model["num_pages"]<LOW_PAGE_COUNT_THRESHOLD).sum()}\n'
    f'books with more than {HIGH_PAGE_COUNT_THRESHOLD} pages : {(df_model["num_pages"]>HIGH_PAGE_COUNT_THRESHOLD).sum()}\n'
    f'books with zero pages : {(df_model["num_pages"] == 0).sum()}'
)
 


#### Inspect zero-page books

In [ ]:
# show first 20 books with zero pages
df_model[df_model["num_pages"] == 0][["title", "authors", "num_pages", "average_rating", "publisher"]].head(10)

Many zero-page records appear to be audio products based on their title or publisher, but the pattern requires an explicit check before treating zero as missing.

#### Check zero-page rating level

In [ ]:
# avegare rating for books with zero pages
print(f"average rating for books with zero pages : {round(df_model[df_model['num_pages'] == 0]['average_rating'].mean(), 2)}")


#### Review zero-page publishers

In [ ]:
# check distinct publishers for books with zero pages
df_model[df_model['num_pages'] == 0]["publisher"].value_counts()


#### Identifying audiobooks


##### Define audiobook text patterns

Using clear title and publisher terms to flag likely audio products.

In [ ]:
# create a regex pattern to identify audiobooks
AUDIO_REGEX_FLAGS = re.IGNORECASE | re.VERBOSE

AUDIO_INDICATOR_PATTERN = r"""
audio(?:books?|go|text)?\b
|
\bcassettes?\b
|
\bcd\s*audio\b
|
\baudio\s*cd\b
|
\bmp3\b
|
\bsound\s+recording\b
|
\blistening\s+library\b
"""


#### Flag likely audiobooks

Searching title and publisher text for audio markers and storing the result in `is_audio`.

In [ ]:
title_publisher_text = (
    df_model["title"].fillna("") + " " + df_model["publisher"].fillna("")
)

df_model["is_audio"] = title_publisher_text.str.contains(
    AUDIO_INDICATOR_PATTERN, regex=True, flags=AUDIO_REGEX_FLAGS, na=False
)

#### Summarize audiobook rows

In [ ]:
# count the number of audiobooks
print(f"Number of audiobooks : {df_model['is_audio'].sum()}")

# average/median number of pages for audiobooks with a page count > 0
print(
    f"Average number of pages for audiobooks with a page count > 0 : {round(df_model[(df_model['is_audio'] == 1) & (df_model['num_pages'] > 0)]['num_pages'].mean(), 1)}")
print(
    f"Median number of pages for audiobooks with a page count > 0 : {round(df_model[(df_model['is_audio'] == 1) & (df_model['num_pages'] > 0)]['num_pages'].median(), 1)}")

# average/median rating for audiobooks
print(f"Average rating for audiobooks : {round(df_model[df_model['is_audio'] == 1]['average_rating'].mean(), 2)}")
print( f"Median rating for audiobooks : {round(df_model[df_model['is_audio'] == 1]['average_rating'].median(), 2)}")

# show first 20 books ordered by num_pages
df_model[df_model["is_audio"] == 1][["title", "authors", "num_pages", "average_rating", "publisher"]].sort_values(by="num_pages", ascending=False).head(10)


Zero pages occur frequently among audio products, but not all audio records have zero pages. One flagged record has more than 1,000 pages, so the text flag is informative rather than a definitive format label.

#### Comparing low-page audio status

In [ ]:
# Check how many books with a page count less than LOW_PAGE_COUNT_THRESHOLD are audiobooks vs non-audiobooks
low_page_count_books = df_model[df_model['num_pages'] < LOW_PAGE_COUNT_THRESHOLD].shape[0]
low_page_count_audio_books = df_model[(df_model['num_pages'] < LOW_PAGE_COUNT_THRESHOLD) & (df_model['is_audio'] == 1)].shape[0]
low_page_count_non_audio_books = df_model[(df_model['num_pages'] < LOW_PAGE_COUNT_THRESHOLD) & (df_model['is_audio'] == 0)].shape[0]

print(f"Total number of books with num_pages < {LOW_PAGE_COUNT_THRESHOLD}: {low_page_count_books}")
print(f"Number of audiobooks with num_pages < {LOW_PAGE_COUNT_THRESHOLD}: {low_page_count_audio_books}")
print(f"Number of non-audiobooks with num_pages < {LOW_PAGE_COUNT_THRESHOLD}: {low_page_count_non_audio_books}")


Publisher text alone does not identify every zero-page record, so non-audio cases also require inspection.

#### Inspecting non-audio zero-page rows

In [ ]:
# Show first 20 books with num_pages = 0 and is_audio = 0
df_model[(df_model["num_pages"] == 0) & (df_model["is_audio"] == 0)][["title", "authors", "num_pages", "average_rating","ratings_count", "publisher"]].head(10)


#### Selecting a publisher to investigate further

In [ ]:
# show first 10 books with a specific publisher
publisher_lookup_value = "basic books"
df_model[df_model["publisher"] == publisher_lookup_value][["title", "authors", "num_pages", "average_rating", "publisher"]].head(10)


Zero-page values are treated as missing rather than literal page counts because both audio and non-audio records are affected.

#### Cleaning books with zero pages


**Decision:** Retain the rows, mark `num_pages = 0` as missing, and fill the missing values with the median valid page count for the audio or non-audio group. Filling missing values from a reference statistic is called imputation.

##### Fill and validate missing page counts

Creating a missing-value flag, applying group medians, and previewing the corrected rows.

In [ ]:
# Flag rows where page count is missing / invalid
df_model["num_pages_missing"] = df_model["num_pages"].eq(0).astype(int)

# Create a cleaned page-count feature
df_model["num_pages_clean"] = df_model["num_pages"].replace(0, np.nan)

audio_page_count_median = round(df_model[df_model["is_audio"] == 1]["num_pages_clean"].median(), 0)
non_audio_page_count_median = round(df_model[df_model["is_audio"] == 0]["num_pages_clean"].median(), 0)

print(f"Median page count for audiobooks : {audio_page_count_median}")
print(f"Median page count for non-audiobooks : {non_audio_page_count_median}")

# Impute with the median of valid page counts for each group
df_model.loc[df_model["is_audio"] == 1, "num_pages_clean"] = df_model.loc[df_model["is_audio"] == 1, "num_pages_clean"].fillna(audio_page_count_median)
df_model.loc[df_model["is_audio"] == 0, "num_pages_clean"] = df_model.loc[df_model["is_audio"] == 0, "num_pages_clean"].fillna(non_audio_page_count_median)

# check the results for books with zero pages
df_model[df_model["num_pages_missing"] == 1][["title", "num_pages", "num_pages_clean", "publisher"]].head(10)


#### Dropping the num_pages column and renaming num_pages_clean to num_pages


In [ ]:
# drop the num_pages column
df_model = df_model.drop(columns=["num_pages"])
# rename num_pages_clean to num_pages
df_model = df_model.rename(columns={"num_pages_clean": "num_pages"})


#### Continuing investigation into high page-count books


In [ ]:
high_page_count_columns = ["title", "num_pages", "average_rating"]
df_model.loc[
    df_model["num_pages"] > HIGH_PAGE_COUNT_THRESHOLD,
    high_page_count_columns,
].head(20)

Most records above 2,000 pages are multi-book sets, although some are not. Their higher average rating is an association and does not show that length causes higher ratings.

#### Looking at the correlation between num_pages and average_rating

In [ ]:
# Compare the cleaned raw page-count feature to average_rating
page_count_correlation_df = df_model[["num_pages", "average_rating"]].copy()

# Pearson: linear relationship
page_count_pearson_corr = page_count_correlation_df["num_pages"].corr(
    page_count_correlation_df["average_rating"], method="pearson")

# Spearman: rank/monotonic relationship, usually better here because pages are skewed
page_count_spearman_corr = page_count_correlation_df["num_pages"].corr(
    page_count_correlation_df["average_rating"], method="spearman")

print("Pearson correlation (cleaned num_pages):", round(page_count_pearson_corr, 4))
print("Spearman correlation (cleaned num_pages):", round(page_count_spearman_corr, 4))
print("Rows used:", len(page_count_correlation_df))


##### Plot the page-count relationship

In [ ]:
# Summarize the continuous page-count relationship in equal-frequency groups.
page_count_plot_df = page_count_correlation_df.copy()
page_count_plot_df["page_quantile"] = pd.qcut(
    page_count_plot_df["num_pages"],
    q=12,
    duplicates="drop",
)
page_count_trend = (
    page_count_plot_df.groupby("page_quantile", observed=True)
    .agg(
        num_pages=("num_pages", "median"),
        median_rating=("average_rating", "median"),
        rating_10=("average_rating", lambda values: values.quantile(0.10)),
        rating_90=("average_rating", lambda values: values.quantile(0.90)),
    )
    .reset_index(drop=True)
)

page_relationship_limit = page_count_trend["num_pages"].max() * 1.05

fig, ax = plt.subplots(figsize=(10, 5.4))
ax.fill_between(
    page_count_trend["num_pages"],
    page_count_trend["rating_10"],
    page_count_trend["rating_90"],
    color=COLOR_BLUE,
    alpha=0.16,
    label="Middle 80% of ratings",
)
ax.plot(
    page_count_trend["num_pages"],
    page_count_trend["median_rating"],
    color=COLOR_BLUE,
    linewidth=2.5,
    marker="o",
    markersize=5,
    label="Median rating",
)
ax.axhline(
    page_count_plot_df["average_rating"].median(),
    color=COLOR_CORAL,
    linestyle="--",
    linewidth=1.7,
    label="Overall median",
)
ax.set(
    xlim=(0, page_relationship_limit),
    title="Typical ratings vary only slightly across page counts",
    xlabel="Median cleaned page count within each equal-frequency group",
    ylabel="Average rating (0–5)",
)
ax.legend(frameon=False, ncol=3, loc="lower right")
ax.grid(axis="x", visible=False)

plt.tight_layout()
plt.show()

The cleaned `num_pages` feature has only a weak relationship with `average_rating`. Page-count bands provide a simpler exploratory comparison.

#### Creating num_pages bands

In [ ]:
page_count_bins = [0, 50, 100, 200, 300, 500, 800, 1_200, np.inf]
page_count_labels = ["1-50", "51-100", "101-200", "201-300", "301-500", "501-800", "801-1200", "1200+"]

df_model["page_count_band"] = pd.cut(
    df_model["num_pages"],
    bins=page_count_bins,
    labels=page_count_labels,
    include_lowest=True,
)

page_count_band_summary = (
    df_model.groupby("page_count_band", observed=True)
    .agg(
        books=("average_rating", "size"),
        mean_rating=("average_rating", "mean"),
        median_rating=("average_rating", "median"),
        rating_std=("average_rating", "std"),
        median_num_pages=("num_pages", "median"),
    )
    .reset_index()
)
page_count_band_summary["share_of_df_model"] = page_count_band_summary["books"] / len(df_model)

page_count_band_summary


##### Plot the page-count band summary

In [ ]:
# Plot how mean average_rating changes across page-count bands.
page_count_band_plot = page_count_band_summary.copy()
page_count_band_plot["ci_95"] = (
    1.96
    * page_count_band_plot["rating_std"]
    / np.sqrt(page_count_band_plot["books"])
)
x_positions = np.arange(len(page_count_band_plot))

fig, ax = plt.subplots(figsize=(11, 5.8))
ax.errorbar(
    x_positions,
    page_count_band_plot["mean_rating"],
    yerr=page_count_band_plot["ci_95"],
    fmt="none",
    ecolor=COLOR_NEUTRAL,
    elinewidth=1.5,
    capsize=4,
    zorder=2,
)
ax.scatter(
    x_positions,
    page_count_band_plot["mean_rating"],
    s=95,
    color=COLOR_PRIMARY,
    edgecolor="white",
    linewidth=1.2,
    zorder=3,
)
ax.axhline(
    df_model["average_rating"].mean(),
    color=COLOR_CORAL,
    linestyle="--",
    linewidth=1.8,
    label="Overall mean",
)
for x_position, row in page_count_band_plot.iterrows():
    ax.text(
        x_position,
        row["mean_rating"] + row["ci_95"] + 0.025,
        f"n={row['books']:,}",
        ha="center",
        va="bottom",
        fontsize=8.5,
    )
ax.set(
    title="Ratings rise most clearly above 500 pages, where samples are smaller",
    xlabel="Cleaned page-count band",
    ylabel="Mean average rating with 95% confidence interval",
    xticks=x_positions,
    xticklabels=page_count_band_plot["page_count_band"],
)
ax.tick_params(axis="x", rotation=25)
ax.legend(frameon=False)
ax.grid(axis="x", visible=False)

plt.tight_layout()
plt.show()

The bands are retained for EDA only. The final modelling pipeline uses continuous page count so it does not discard within-band differences.

### 2.6 `text_reviews_count`


#### First look


In [ ]:
# Describe text_reviews_count
df_model["text_reviews_count"].describe()

##### Plot the text-review distribution

In [ ]:
text_reviews_plot_cutoff = df_model["text_reviews_count"].quantile(0.95)
text_reviews_for_plot = df_model.loc[
    df_model["text_reviews_count"] <= text_reviews_plot_cutoff,
    "text_reviews_count",
]
log_text_reviews_for_plot = np.log1p(df_model["text_reviews_count"])
median_text_reviews = df_model["text_reviews_count"].median()

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
sns.histplot(
    text_reviews_for_plot,
    bins=42,
    stat="percent",
    color=COLOR_PRIMARY,
    alpha=0.82,
    ax=axes[0],
)
axes[0].axvline(
    median_text_reviews,
    color=COLOR_CORAL,
    linestyle="--",
    linewidth=2,
    label=f"Median: {median_text_reviews:.0f}",
)
axes[0].set(
    title="Raw scale: the long tail dominates",
    xlabel=f"Written reviews (through 95th percentile: {text_reviews_plot_cutoff:.0f})",
    ylabel="Share of books",
)
axes[0].yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
axes[0].legend(frameon=False)
axes[0].grid(axis="x", visible=False)

sns.histplot(
    log_text_reviews_for_plot,
    bins=42,
    stat="percent",
    color=COLOR_BLUE,
    alpha=0.82,
    ax=axes[1],
)
axes[1].axvline(
    np.log1p(median_text_reviews),
    color=COLOR_CORAL,
    linestyle="--",
    linewidth=2,
)
axes[1].set(
    title="Log scale: low and medium activity becomes visible",
    xlabel="log1p(written reviews)",
    ylabel="Share of books",
)
axes[1].yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
axes[1].grid(axis="x", visible=False)

fig.suptitle("Written-review activity is strongly right-skewed", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(
    "Books above the 95th-percentile plot cutoff: "
    f"{(df_model['text_reviews_count'] > text_reviews_plot_cutoff).sum()}"
)

More than half of the books have fewer than 100 written reviews, and the distribution is strongly right-skewed. A logarithmic version compresses the largest values and is used for the linear model.

#### Inspecting most-reviewed books

In [ ]:
# first 10 books, ordered by text_reviews_count
df_model[["title", "authors", "text_reviews_count", "average_rating", "ratings_count"]].sort_values(by="text_reviews_count", ascending=False).head(10)


#### Inspecting books with no text reviews

In [ ]:
# zero reviews count
zero_text_review_books = df_model.loc[df_model["text_reviews_count"] == 0][["title", "authors", "average_rating", "ratings_count"]]
zero_text_review_books.head(10)


##### Summarize zero-review ratings

In [ ]:
# average rating for books with zero reviews
print(f"average rating for books with zero reviews : {round(zero_text_review_books['average_rating'].mean(), 2)}")


Books with no written reviews have an average rating close to the overall dataset mean, so the zero-review indicator is not a strong signal by itself.

Because review counts span several orders of magnitude, the relationship is summarized on a logarithmic count scale using equal-frequency groups.

#### Relate written-review volume to rating

In [ ]:
# Summarize the review-count relationship on a readable logarithmic scale.
text_reviews_correlation_df = df_model[["text_reviews_count", "average_rating"]].copy()
text_reviews_correlation_df["text_reviews_count_log"] = np.log1p(
    text_reviews_correlation_df["text_reviews_count"]
)
text_reviews_correlation_df["review_quantile"] = pd.qcut(
    text_reviews_correlation_df["text_reviews_count_log"],
    q=15,
    duplicates="drop",
)
text_review_trend = (
    text_reviews_correlation_df.groupby("review_quantile", observed=True)
    .agg(
        text_reviews_count_log=("text_reviews_count_log", "median"),
        median_rating=("average_rating", "median"),
        rating_10=("average_rating", lambda values: values.quantile(0.10)),
        rating_90=("average_rating", lambda values: values.quantile(0.90)),
    )
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(10, 5.4))
ax.fill_between(
    text_review_trend["text_reviews_count_log"],
    text_review_trend["rating_10"],
    text_review_trend["rating_90"],
    color=COLOR_BLUE,
    alpha=0.16,
    label="Middle 80% of ratings",
)
ax.plot(
    text_review_trend["text_reviews_count_log"],
    text_review_trend["median_rating"],
    color=COLOR_BLUE,
    linewidth=2.5,
    marker="o",
    markersize=5,
    label="Median rating",
)
ax.axhline(
    text_reviews_correlation_df["average_rating"].median(),
    color=COLOR_CORAL,
    linestyle="--",
    linewidth=1.7,
    label="Overall median",
)
review_tick_counts = [0, 10, 100, 1_000, 10_000, 100_000]
review_tick_counts = [
    count
    for count in review_tick_counts
    if count <= text_reviews_correlation_df["text_reviews_count"].max()
]
ax.set_xticks(np.log1p(review_tick_counts))
ax.set_xticklabels([f"{count:,}" for count in review_tick_counts])
ax.set(
    title="Typical ratings change little across written-review volume",
    xlabel="Number of written reviews (log scale)",
    ylabel="Average rating (0–5)",
)
ax.legend(frameon=False, ncol=3, loc="lower right")
ax.grid(axis="x", visible=False)

plt.tight_layout()
plt.show()

The median rating remains close to the overall median across review-volume groups. The middle 80% narrows only modestly, reinforcing that written-review count is a weak standalone signal rather than evidence of a causal effect.

### 2.7 `language_code`


#### First look


In [ ]:
language_value_counts = df_model["language_code"].value_counts(dropna=False)
language_value_counts

Several language codes are rare. Grouping them avoids creating many categories supported by only a few books.

#### Compare ratings by language

In [ ]:
# Plot frequent language groups and a combined rare-language group.
LANGUAGE_PLOT_MIN_BOOKS = 50

language_value_counts = df_model["language_code"].value_counts(dropna=False)
common_language_codes_for_plot = language_value_counts[
    language_value_counts >= LANGUAGE_PLOT_MIN_BOOKS
].index

language_rating_plot_data = df_model[["language_code", "average_rating"]].copy()
language_rating_plot_data["language_group"] = (
    language_rating_plot_data["language_code"]
    .where(
        language_rating_plot_data["language_code"].isin(
            common_language_codes_for_plot
        ),
        "other_language",
    )
)
language_rating_plot_df = (
    language_rating_plot_data.groupby("language_group", dropna=False)
    .agg(
        books=("average_rating", "size"),
        mean_rating=("average_rating", "mean"),
        rating_std=("average_rating", "std"),
    )
    .reset_index()
    .sort_values("mean_rating")
)
language_rating_plot_df["ci_95"] = (
    1.96
    * language_rating_plot_df["rating_std"]
    / np.sqrt(language_rating_plot_df["books"])
)
language_rating_plot_df["plot_label"] = language_rating_plot_df[
    "language_group"
].replace({"other_language": "other_language (combined)"})

y_positions = np.arange(len(language_rating_plot_df))
language_point_colors = [
    COLOR_ACCENT if language_group == "other_language" else COLOR_PRIMARY
    for language_group in language_rating_plot_df["language_group"]
]

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.errorbar(
    language_rating_plot_df["mean_rating"],
    y_positions,
    xerr=language_rating_plot_df["ci_95"],
    fmt="none",
    ecolor=COLOR_NEUTRAL,
    elinewidth=1.5,
    capsize=4,
    zorder=2,
)
ax.scatter(
    language_rating_plot_df["mean_rating"],
    y_positions,
    s=95,
    color=language_point_colors,
    edgecolor="white",
    linewidth=1.2,
    zorder=3,
)
ax.axvline(
    df_model["average_rating"].mean(),
    color=COLOR_CORAL,
    linestyle="--",
    linewidth=1.8,
    label="Overall mean",
)
for y_position, row in language_rating_plot_df.reset_index(drop=True).iterrows():
    ax.text(
        row["mean_rating"] + row["ci_95"] + 0.008,
        y_position,
        f"n={row['books']:,}",
        va="center",
        fontsize=8.5,
    )
ax.set(
    title="Frequent language groups have similar mean ratings",
    xlabel="Mean average rating with 95% confidence interval",
    ylabel="Language code or combined rare-language group",
    yticks=y_positions,
    yticklabels=language_rating_plot_df["plot_label"],
)
ax.legend(frameon=False)
ax.grid(axis="y", visible=False)

plt.tight_layout()
plt.show()

The frequent language codes remain close to the overall mean. The combined `other_language` group is higher, but it pools several heterogeneous language codes with small individual samples and should not be interpreted as one coherent language effect.

#### Finding multi-language titles

In [ ]:
# check for multi-language books
multi_language_books = (
    df_model.groupby(["title", "authors", "average_rating"])["language_code"]
    .agg(n_languages="nunique", languages=lambda language_codes: sorted(language_codes.dropna().unique()))
    .reset_index()
    .query("n_languages > 1")
    .sort_values("n_languages", ascending=False)
    
)
multi_language_books.head(10)

##### Count multi-language books

In [ ]:
# number of multi-language books
print("number of multi-language books :", len(multi_language_books))

Most books are not identified as multi-language. Many matches are English variants rather than genuinely multilingual editions, which limits the usefulness of this exploratory flag.

#### Average rating for multi-language books

In [ ]:
# average rating for multi-language books
round(multi_language_books["average_rating"].mean(), 2)


The multi-language subset has an average rating close to the full dataset. The final pipeline therefore uses the grouped language code rather than this title-based flag.

#### Modelling: group rare languages

Keeping frequent language codes and combining the rest into `other_language` avoids a sparse categorical feature with many poorly represented categories.

In [ ]:
# Create a grouped language-code feature
LANGUAGE_GROUP_MIN_COUNT = 50

language_counts_for_grouping = df_model["language_code"].value_counts(dropna=False)
common_language_codes = language_counts_for_grouping[
    language_counts_for_grouping >= LANGUAGE_GROUP_MIN_COUNT
].index

df_model["language_code_grouped"] = df_model["language_code"].where(
    df_model["language_code"].isin(common_language_codes),
    "other_language",
)

language_grouped_summary = (
    df_model.groupby("language_code_grouped")
    .agg(
        books=("average_rating", "size"),
        mean_rating=("average_rating", "mean"),
        median_rating=("average_rating", "median"),
    )
    .reset_index()
    .sort_values("books", ascending=False)
)
language_grouped_summary["share_of_df_model"] = language_grouped_summary["books"] / len(df_model)
print("Language codes kept as their own group:", sorted(common_language_codes))

language_grouped_summary


The grouped language feature keeps frequent language codes as separate categories and combines rare codes into `other_language`. Most well represented language have an average rating close to the average rating of the dataset, but for rare languages, the average rating is significantly higher.


### 2.8 `authors`


Author strings use `/` to separate contributors. Counting all contributors and keeping the first listed author provides two compact candidate features.

#### Count authors per book

In [ ]:
# check what is the maximum number of authors for one book
author_count_per_book = df_model["authors"].str.split("/").str.len()
print("max number of authors for one book:", author_count_per_book.max())


#### Inspect the maximum-author example

The maximum is 51 listed authors, so inspecting that record checks whether the count reflects the source string.

In [ ]:
author_preview_columns = ["title", "authors"]
df_model.loc[author_count_per_book == author_count_per_book.max(), author_preview_columns]


**Decision:** Create `first_author` as a compact main-author feature and `n_authors` as the number of listed contributors.

#### Creating author features

In [ ]:
# creating new columns
df_model["first_author"] = df_model["authors"].str.split(
    "/").str[0].str.strip()
df_model["n_authors"] = author_count_per_book


#### Plot author-count distribution

In [ ]:
# Plot the distribution of author counts to show how many authors most books have.
author_count_bucket_order = ["1", "2", "3", "4–5", "6–10", "11+"]
author_count_plot_df = df_model[["n_authors", "average_rating"]].copy()
author_count_plot_df["author_count_bucket"] = pd.cut(
    author_count_plot_df["n_authors"],
    bins=[0, 1, 2, 3, 5, 10, np.inf],
    labels=author_count_bucket_order,
    include_lowest=True,
)
author_count_distribution = (
    author_count_plot_df["author_count_bucket"]
    .value_counts(sort=False)
    .reindex(author_count_bucket_order)
    .rename_axis("author_count_bucket")
    .reset_index(name="books")
)
author_count_distribution["share_percent"] = (
    author_count_distribution["books"] / len(author_count_plot_df) * 100
)

fig, ax = plt.subplots(figsize=(10, 5.2))
author_bars = sns.barplot(
    data=author_count_distribution,
    x="author_count_bucket",
    y="share_percent",
    color=COLOR_PRIMARY,
    ax=ax,
)
for bar, share, books in zip(
    author_bars.patches,
    author_count_distribution["share_percent"],
    author_count_distribution["books"],
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{share:.1f}%\n(n={books:,})",
        ha="center",
        va="bottom",
        fontsize=8.5,
    )
ax.set(
    title="Single-author books dominate the dataset",
    xlabel="Number of listed authors",
    ylabel="Share of books",
)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
ax.grid(axis="x", visible=False)

plt.tight_layout()
plt.show()

Books with more than ten authors are rare, and most books have one listed author.

#### Full author list vs first author comparison

In [ ]:
# Print number of unique authors vs first author
print(f"Number of unique authors : {df_model['authors'].nunique()}")
print(f"Number of unique first authors : {df_model['first_author'].nunique()}")


#### Checking for rating patterns based on the number of authors

In [ ]:
# Compare how ratings vary across readable author-count groups.
author_rating_summary = (
    author_count_plot_df.groupby("author_count_bucket", observed=True)
    .agg(
        books=("average_rating", "size"),
        mean_rating=("average_rating", "mean"),
        rating_std=("average_rating", "std"),
    )
    .reindex(author_count_bucket_order)
    .reset_index()
)
author_rating_summary["ci_95"] = (
    1.96
    * author_rating_summary["rating_std"]
    / np.sqrt(author_rating_summary["books"])
)
x_positions = np.arange(len(author_rating_summary))

fig, ax = plt.subplots(figsize=(10, 5.4))
ax.errorbar(
    x_positions,
    author_rating_summary["mean_rating"],
    yerr=author_rating_summary["ci_95"],
    fmt="none",
    ecolor=COLOR_NEUTRAL,
    elinewidth=1.5,
    capsize=4,
    zorder=2,
)
ax.scatter(
    x_positions,
    author_rating_summary["mean_rating"],
    s=95,
    color=COLOR_GOLD,
    edgecolor="white",
    linewidth=1.2,
    zorder=3,
)
ax.axhline(
    df_model["average_rating"].mean(),
    color=COLOR_CORAL,
    linestyle="--",
    linewidth=1.8,
    label="Overall mean",
)
overall_author_mean = df_model["average_rating"].mean()
for x_position, row in author_rating_summary.iterrows():
    label_above = row["mean_rating"] >= overall_author_mean
    label_y = (
        row["mean_rating"] + row["ci_95"] + 0.018
        if label_above
        else row["mean_rating"] - row["ci_95"] - 0.018
    )
    ax.text(
        x_position,
        label_y,
        f"n={row['books']:,}",
        ha="center",
        va="bottom" if label_above else "top",
        fontsize=8.5,
    )
ax.set(
    title="Rare multi-author groups have wide rating uncertainty",
    xlabel="Number of listed authors",
    ylabel="Mean average rating with 95% confidence interval",
    xticks=x_positions,
    xticklabels=author_rating_summary["author_count_bucket"],
)
ax.legend(frameon=False)
ax.grid(axis="x", visible=False)

plt.tight_layout()
plt.show()

No clear correlation between number of authors and average rating.


#### Modelling authors features


Author identity has many distinct values. Frequency encoding records how often the first author appears, while target encoding replaces an author with a training-only summary of that author's ratings.

#### Creating author frequency feature

In [ ]:
# Frequency encoding candidate for author information : how many rows share the same first_author.
first_author_frequency_counts = df_model["first_author"].value_counts()

df_model["first_author_frequency"] = (
    df_model["first_author"].map(first_author_frequency_counts).fillna(0).astype(int)
)

author_feature_preview_columns = [
    "first_author",
    "authors",
    "n_authors",
    "first_author_frequency",
]

df_model[author_feature_preview_columns].head()


#### Author target encoding

Target encoding replaces a category with a smoothed average of `average_rating`. Smoothing pulls authors with few books toward the overall mean so they are not over-trusted. This EDA version is descriptive; the modelling version is fitted on training rows only.

#### Helper function for target encoding (Used for publisher and first_author)

In [ ]:
# Exploratory target encoding for first_author.
# Smoothing pulls rare authors toward the global mean so one-book authors are not over-trusted.
TARGET_ENCODING_SMOOTHING = 10

def add_smoothed_target_encoding(data, category_column, encoded_column, count_column):
    global_target_mean = data["average_rating"].mean()
    mean_column = f"{category_column}_mean_rating"

    target_summary = (
        data.groupby(category_column, dropna=False)
        .agg(
            **{
                count_column: ("average_rating", "size"),
                mean_column: ("average_rating", "mean"),
            }
        )
    )
    target_summary[encoded_column] = (
        (
            target_summary[count_column] * target_summary[mean_column]
            + TARGET_ENCODING_SMOOTHING * global_target_mean
        )
        / (target_summary[count_column] + TARGET_ENCODING_SMOOTHING)
    )

    data[encoded_column] = (
        data[category_column]
        .map(target_summary[encoded_column])
        .fillna(global_target_mean)
    )

    return target_summary.reset_index()

##### Apply target encoding to first author

Applying the helper and showing the authors with the most books.

In [ ]:
first_author_target_summary = add_smoothed_target_encoding(
    df_model,
    category_column="first_author",
    encoded_column="first_author_target_mean",
    count_column="author_books",
)

first_author_target_summary.sort_values(
    "author_books", ascending=False).head(15).round(3)

#### Check author features correlation with average_ratings

In [ ]:
# Spearman correlation with average_rating for the selected numeric author features.
author_numeric_features = [
    "n_authors",
    "first_author_frequency",
    "first_author_target_mean",
]

author_feature_correlation = (
    df_model[author_numeric_features + ["average_rating"]]
    .corr(method="spearman")
    .loc[author_numeric_features, "average_rating"]
    .reset_index()
    .rename(columns={"index": "feature", "average_rating": "spearman_corr"})
    .assign(abs_spearman=lambda data: data["spearman_corr"].abs())
    .sort_values("abs_spearman", ascending=False)
    .drop(columns="abs_spearman")
)

author_feature_correlation.round(4)

`n_authors` and `first_author_frequency` have weak order-based correlations with `average_rating`. The exploratory target mean is stronger because it is calculated from ratings, so the final version must be fitted on training data only to avoid leakage from the test set.

### 2.9 `title`


#### Duplicates


Unique identifiers show that the source has no exact identifier duplicates. Repeated titles can still represent different editions, languages, publishers, or collections.

#### Title duplicates

In [ ]:
# Reuse title frequencies throughout the duplicate-title analysis.
title_frequencies = df_model["title"].value_counts()
title_frequencies


#### Inspect one repeated title

Using `the iliad` as an example to see how editions and author strings differ for the same title.


In [ ]:
iliad_preview_columns = [
    "title",
    "first_author",
    "language_code",
    "publisher",
    "publication_year",
    "average_rating",
]
df_model.loc[df_model["title"].eq("the iliad"), iliad_preview_columns].head(10)

The author strings vary across these editions while the first author remains the same. This supports `first_author` as a compact feature, not as a unique edition identifier.

#### Duplicates by Title and authors

In [ ]:
# One consolidated summary for repeated titles and their author variants.
title_author_summary = (
    df_model.groupby("title")
    .agg(
        n_books=("title", "size"),
        n_author_strings=("authors", "nunique"),
        n_first_authors=("first_author", "nunique"),
    )
)

repeated_titles = title_frequencies[title_frequencies.gt(1)].index
titles_with_different_first_author = title_author_summary.index[
    title_author_summary["n_first_authors"].gt(1)
]
duplicate_title_author_pairs = df_model.groupby(["title", "authors"]).size().gt(1).sum()
duplicate_title_first_author_pairs = (
    df_model.groupby(["title", "first_author"]).size().gt(1).sum()
)

print(f"Repeated titles: {len(repeated_titles)}")
print(f"Repeated title + author pairs: {duplicate_title_author_pairs}")
print(f"Repeated title + first-author pairs: {duplicate_title_first_author_pairs}")
print("Repeated titles with more than one first author: "f"{len(titles_with_different_first_author)}")


#### Inspect title conflicts


In [ ]:
# Show the title/first-author combinations for repeated titles with multiple first authors.
title_first_author_groups = (
    df_model.loc[df_model["title"].isin(titles_with_different_first_author)]
    .groupby(["title", "first_author"])
    .size()
    .reset_index(name="count")
    .sort_values(["title", "count"], ascending=[True, False])
    .head(20)
)
title_first_author_groups

#### Inspect rating variants within same-title first-author groups

Aggregate each title/first-author group once to identify cases where the group contains different ratings.


In [ ]:
# Inspect same-title, same-first-author groups that have more than one rating.
title_first_author_rating_variants = (
    df_model.groupby(["title", "first_author"])
    .agg(
        n_unique_average_rating=("average_rating", "nunique"),
        ratings_list=("average_rating", lambda ratings: list(ratings.unique())),
        num_pages_list=("num_pages", lambda page_counts: list(page_counts.unique())),
        language_code_list=("language_code", lambda language_codes: list(language_codes.unique())),
        publisher_list=("publisher", lambda publishers: list(publishers.unique())),
        publication_year_list=("publication_year", lambda publication_years: list(publication_years.unique())),
    )
    .reset_index()
    .query("n_unique_average_rating > 1")
)

print(
    "Same-title, same-first-author groups with different average ratings: "
    f"{len(title_first_author_rating_variants)}"
)
title_first_author_rating_variants.head(20)


The rating variants show that `title` plus `first_author` is not a reliable duplicate key: the records can differ in edition-level fields such as publisher, language, page count, and publication year.


**[Decision]** Keep these rows as they are. Repeated titles may represent different editions, publishers, languages, or collections.


#### Collections and series


##### Define collection and series patterns

Using visible title markers to create separate collection and series flags.

In [ ]:
# Add title-based flags for collections and series entries.
# The identifiers for series are "#[number]" / "Volume [number]" / "Vol. [number]" / "Books [number]" / "Part [number]" / "Tome [number]".
TITLE_REGEX_FLAGS = re.IGNORECASE | re.VERBOSE

SERIES_NUMBER_PATTERN = r"(?:\d+(?:\.\d+)?|[ivxlcdm]+|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve)"
SERIES_NUMBER_RANGE_PATTERN = rf"{SERIES_NUMBER_PATTERN}(?:\s*(?:[-\u2013\u2014,/&]|and)\s*{SERIES_NUMBER_PATTERN})*"

collection_title_pattern = r"""
\b(?:box(?:ed)?\s+set|omnibus|antholog(?:y|ies)|collection)\b
|
\b(?:complete|collected|selected|essential|major)\s+
(?:works|novels|stories|short\s+stories|poems|poetry|plays|writings|letters|essays|tragedies|adventures|memoirs)\b
|
\b\d+\s*(?:book|volume)s?\s+(?:box(?:ed)?\s+set|set|collection)\b
|
\b(?:\d+\s+)?volumes?\s+set\b
"""

series_title_pattern = rf"""
\([^)]*(?:\#\s*{SERIES_NUMBER_RANGE_PATTERN}|\b(?:vol(?:ume)?s?\.?|books?|part|tome)\s+{SERIES_NUMBER_RANGE_PATTERN})[^)]*\)
|
(?<!\w)\#\s*{SERIES_NUMBER_RANGE_PATTERN}\b
|
\b(?:vol(?:ume)?s?\.?|part|tome)\s+{SERIES_NUMBER_RANGE_PATTERN}\b
"""

title_series = df_model["title"].fillna("")

df_model["is_collection"] = title_series.str.contains(
    collection_title_pattern, regex=True, flags=TITLE_REGEX_FLAGS, na=False)
df_model["is_series"] = title_series.str.contains(
    series_title_pattern, regex=True, flags=TITLE_REGEX_FLAGS, na=False)


#### Series and collections average ratings

In [ ]:
# Compare the rating level of books flagged as collections or series entries.
# The comparison uses both mean and median because average_rating is bounded and slightly skewed.
collection_series_summary_rows = []

for label, flag_column in {
    "Collections": "is_collection",
    "Series entries": "is_series",
}.items():
    flagged_books = df_model[df_model[flag_column]]
    unflagged_books = df_model[~df_model[flag_column]]

    collection_series_summary_rows.append({
        "category": label,
        "books": len(flagged_books),
        "share_of_df_model": len(flagged_books) / len(df_model),
        "mean_rating": flagged_books["average_rating"].mean(),
        "median_rating": flagged_books["average_rating"].median(),
        "non_flagged_mean_rating": unflagged_books["average_rating"].mean(),
        "mean_difference": flagged_books["average_rating"].mean() - unflagged_books["average_rating"].mean(),
    })

collection_series_summary_df = pd.DataFrame(collection_series_summary_rows)
collection_series_summary_df.round(3)


#### Visualize collection and series rating gaps

In [ ]:
collection_series_plot = collection_series_summary_df.sort_values(
    "mean_difference"
).reset_index(drop=True)
y_positions = np.arange(len(collection_series_plot))

fig, ax = plt.subplots(figsize=(10, 4.4))
ax.hlines(
    y_positions,
    collection_series_plot["non_flagged_mean_rating"],
    collection_series_plot["mean_rating"],
    color=COLOR_LIGHT,
    linewidth=7,
    zorder=1,
)
ax.scatter(
    collection_series_plot["non_flagged_mean_rating"],
    y_positions,
    s=110,
    color=COLOR_NEUTRAL,
    label="Books not flagged",
    zorder=2,
)
ax.scatter(
    collection_series_plot["mean_rating"],
    y_positions,
    s=140,
    color=COLOR_PRIMARY,
    label="Flagged books",
    zorder=3,
)
for y_position, row in collection_series_plot.iterrows():
    ax.text(
        row["mean_rating"] + 0.012,
        y_position,
        f"+{row['mean_difference']:.2f}  ·  n={row['books']:,}",
        va="center",
        fontsize=9,
    )
ax.set(
    title="Collections show the larger descriptive rating gap",
    xlabel="Mean average rating",
    ylabel="",
    yticks=y_positions,
    yticklabels=collection_series_plot["category"],
)
ax.legend(frameon=False, ncol=2, loc="lower right")
ax.grid(axis="y", visible=False)

plt.tight_layout()
plt.show()

Collections represent a small share of `df_model`, so their higher mean and median rating should be interpreted carefully. The pattern is still plausible: collections, complete works, boxed sets, and selected works are often created for books or authors with an existing audience.

Series entries are much more common. Their mean rating is only slightly higher than non-series books, so `is_series` may add some signal, but it is unlikely to be a strong standalone predictor.

### 2.10 `publisher`


#### First look

Publisher was already used earlier in the audiobook processing: `is_audio` was derived from title and publisher text because several low or zero page-count records looked like audiobook publishers.


#### Count publishers

Counting publisher frequencies to understand how sparse this categorical field is.


In [ ]:
publisher_value_counts = df_model["publisher"].value_counts(dropna=False)
publisher_value_counts.head(20)


#### Plot most common publishers

In [ ]:
TOP_PUBLISHER_PLOT_COUNT = 15

top_publisher_count_df = (
    publisher_value_counts
    .head(TOP_PUBLISHER_PLOT_COUNT)
    .rename_axis("publisher")
    .reset_index(name="books")
    .sort_values("books")
)
top_publisher_count_df["share_percent"] = (
    top_publisher_count_df["books"] / len(df_model) * 100
)

fig, ax = plt.subplots(figsize=(10, 6.5))
publisher_bars = ax.barh(
    top_publisher_count_df["publisher"],
    top_publisher_count_df["share_percent"],
    color=COLOR_PRIMARY,
    alpha=0.88,
)
ax.bar_label(
    publisher_bars,
    labels=[
        f"{share:.1f}%  ·  {books:,}"
        for share, books in zip(
            top_publisher_count_df["share_percent"],
            top_publisher_count_df["books"],
        )
    ],
    padding=4,
    fontsize=8.5,
)
ax.set(
    title="Even the most frequent publisher represents only 3% of books",
    xlabel="Share of books  ·  count",
    ylabel="Publisher",
)
ax.xaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=1))
ax.grid(axis="y", visible=False)

plt.tight_layout()
plt.show()

The publisher column has many rare values. One-hot encoding, which creates a separate 0/1 column for each category, would therefore produce many mostly empty columns.

#### Profile top publishers

In [ ]:
TOP_PUBLISHER_COUNT = 15

top_publisher_names = publisher_value_counts.head(TOP_PUBLISHER_COUNT).index

publisher_summary = (
    df_model[df_model["publisher"].isin(top_publisher_names)]
    .groupby("publisher")
    .agg(
        books=("average_rating", "size"),
        mean_rating=("average_rating", "mean"),
        median_rating=("average_rating", "median"),
        rating_std=("average_rating", "std"),
        audiobook_share=("is_audio", "mean"),
        collection_share=("is_collection", "mean"),
        series_share=("is_series", "mean"),
    )
    .sort_values("books", ascending=False)
    .reset_index()
)
publisher_summary["share_of_df_model"] = publisher_summary["books"] / len(df_model)

publisher_summary.round(3)


#### Visual profile of leading publishers

Aligned panels separate rating estimates from catalog composition. Sample sizes are labelled directly, and the series share is shown on its own percentage scale.

In [ ]:
publisher_profile_plot = publisher_summary.sort_values("mean_rating").reset_index(drop=True)
publisher_profile_plot["ci_95"] = (
    1.96
    * publisher_profile_plot["rating_std"]
    / np.sqrt(publisher_profile_plot["books"])
)
y_positions = np.arange(len(publisher_profile_plot))

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 7),
    sharey=True,
    gridspec_kw={"width_ratios": [1.7, 1]},
)
axes[0].errorbar(
    publisher_profile_plot["mean_rating"],
    y_positions,
    xerr=publisher_profile_plot["ci_95"],
    fmt="none",
    ecolor=COLOR_NEUTRAL,
    elinewidth=1.3,
    capsize=3,
    zorder=2,
)
axes[0].scatter(
    publisher_profile_plot["mean_rating"],
    y_positions,
    s=95,
    color=COLOR_PRIMARY,
    edgecolor="white",
    linewidth=1.1,
    zorder=3,
)
axes[0].axvline(
    df_model["average_rating"].mean(),
    color=COLOR_CORAL,
    linestyle="--",
    linewidth=1.7,
    label="Overall mean",
)
for y_position, row in publisher_profile_plot.iterrows():
    axes[0].text(
        row["mean_rating"] + row["ci_95"] + 0.008,
        y_position,
        f"n={row['books']:,}",
        va="center",
        fontsize=8,
    )
axes[0].set(
    title="Mean rating",
    xlabel="Mean average rating with 95% confidence interval",
    ylabel="Publisher",
    yticks=y_positions,
    yticklabels=publisher_profile_plot["publisher"],
)
axes[0].legend(frameon=False, loc="lower right")
axes[0].grid(axis="y", visible=False)

series_bars = axes[1].barh(
    y_positions,
    publisher_profile_plot["series_share"],
    color=COLOR_BLUE,
    alpha=0.85,
)
axes[1].bar_label(
    series_bars,
    labels=[f"{share:.0%}" for share in publisher_profile_plot["series_share"]],
    padding=4,
    fontsize=8,
)
axes[1].set(
    title="Series entries in catalog",
    xlabel="Share of publisher's books",
    yticks=y_positions,
)
axes[1].xaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
axes[1].tick_params(axis="y", labelleft=False)
axes[1].grid(axis="y", visible=False)

fig.suptitle(
    "Top publishers differ in rating and series-heavy catalog mix",
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
plt.show()

`viz media llc` has the highest mean rating among the most frequent publishers and an unusually series-heavy catalog. The chart suggests that publisher-level rating differences can partly reflect the kinds of books each publisher releases rather than publisher identity alone.

#### Publisher frequency feature

Publisher frequency records how often each publisher appears without creating thousands of separate columns.

In [ ]:
publisher_frequency_counts = df_model["publisher"].value_counts()

df_model["publisher_frequency"] = (df_model["publisher"].map(publisher_frequency_counts).fillna(0).astype(int))

publisher_feature_preview_columns = [
    "publisher",
    "publisher_frequency",
    "average_rating",
]

df_model[publisher_feature_preview_columns].head()

#### Publisher target encoding

Creating a smoothed average rating for each publisher. The final modelling version is learned from training rows only.

In [ ]:
# Exploratory target encoding for publisher.
publisher_target_summary = add_smoothed_target_encoding(
    df_model,
    category_column="publisher",
    encoded_column="publisher_target_mean",
    count_column="publisher_books",
)

publisher_target_summary.sort_values("publisher_books", ascending=False).head(15).round(3)

#### Publisher feature correlation with average_rating

In [ ]:
# Spearman correlation with average_rating for the selected numeric publisher features.
publisher_numeric_features = [
    "publisher_frequency",
    "publisher_target_mean",
]

publisher_feature_correlation = (
    df_model[publisher_numeric_features + ["average_rating"]]
    .corr(method="spearman")
    .loc[publisher_numeric_features, "average_rating"]
    .reset_index()
    .rename(columns={"index": "feature", "average_rating": "spearman_corr"})
)

publisher_feature_correlation.round(4)

`publisher_frequency` is a simple candidate feature for the first modelling pass, but by itself it does not have a strong relationship with average rating. `publisher_target_mean` is more informative because it uses the publisher's historical average rating, although it is less of a signal than `first_author_target_mean`.

------------
## 3. Modelling

Preparing one reproducible train/test split, building features from the training data, and comparing six regression models on the same held-out test rows. Regression predicts a numeric value, which is the book's `average_rating` in this project.

- **Linear Regression**: fits a straight-line relationship between the input features and average rating.
- **HistGradientBoostingRegressor**: builds a sequence of decision trees, with each tree correcting earlier errors.
- **CatBoostRegressor**: builds boosted trees and can use categorical values such as author or publisher directly.
- **XGBRegressor**: builds boosted trees with controls that limit overfitting.
- **RandomForestRegressor**: averages many decision trees trained on resampled data.
- **ExtraTreesRegressor**: averages highly randomized decision trees.

The models are evaluated with:

- **Mean Absolute Error (MAE)**: average absolute prediction error in rating points. Lower is better.
- **Root Mean Squared Error (RMSE)**: prediction error that gives more weight to large mistakes. Lower is better.
- **R-squared (R2)**: the share of target variation explained by the model. Higher is better.
- **Constant mean baseline**: a simple reference that predicts the training-set mean for every book.

The complete workflow remains visible in the notebook. Model export later refits the selected configuration after the comparison is complete.

### 3.1 Modelling setup

#### Import modelling tools

In [ ]:
# Import the modelling tools used in this section.
# These imports stay in the notebook because the professor should be able to see the full
# modelling workflow without opening an external preprocessing file.
from itertools import product

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import plotly.graph_objects as go
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


### 3.2 Modelling source table

Rebuilding the modelling inputs from cleaned source columns ensures that every learned transformation is fitted only after the train/test split. The final table requires at least 50 ratings per book, a stricter support rule than the ten-rating EDA cutoff.

#### Build the modelling source table

Defining the target, raw input columns, and final rating-support threshold before splitting the data.

In [ ]:
TARGET = "average_rating"

# Build the raw modelling table from cleaned source columns only.
# The row filters match decisions justified in EDA: remove books with very few ratings
# `publication_year` is not included here because it is engineered from `publication_date` later.
RAW_MODEL_INPUT_COLUMNS = [
    "title",
    "authors",
    "language_code",
    "num_pages",
    "ratings_count",
    "text_reviews_count",
    "publication_date",
    "publisher",
]
MIN_RATINGS_COUNT_FOR_MODELLING = 50

model_source_df = df_clean.copy()
model_source_df = model_source_df[model_source_df["ratings_count"] >= MIN_RATINGS_COUNT_FOR_MODELLING].copy()

X_model_raw = model_source_df[RAW_MODEL_INPUT_COLUMNS].copy()
y_model = model_source_df[TARGET].copy()

model_dataset_summary = pd.DataFrame([
    {"metric": "rows", "value": len(model_source_df)},
    {"metric": "raw_input_columns", "value": len(RAW_MODEL_INPUT_COLUMNS)},
    {"metric": "minimum_ratings_count", "value": MIN_RATINGS_COUNT_FOR_MODELLING},
])

model_dataset_summary


### 3.3 Train/test split


#### Create the train/test split

Using rating bands to keep a similar target distribution in both partitions. This is called stratification.

In [ ]:
# Create one fixed stratified split for every model configuration.
# Stratification keeps the concentrated average_rating distribution similar in train and test.
TEST_SIZE = 0.15
RATING_BAND_COUNT = 10
RANDOM_STATE = 42

def create_stratified_rating_split(target):
    rating_band_intervals = pd.qcut(
        target,
        q=RATING_BAND_COUNT,
        duplicates="drop",
    )
    rating_band_codes = rating_band_intervals.cat.codes

    train_index, test_index = train_test_split(
        target.index,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=rating_band_codes,
    )

    return train_index, test_index, rating_band_intervals


MODEL_TRAIN_INDEX, MODEL_TEST_INDEX, model_rating_bands = create_stratified_rating_split(y_model)

split_check_df = pd.concat([
    pd.DataFrame({
        "split": "train",
        TARGET: y_model.loc[MODEL_TRAIN_INDEX],
        "rating_band": model_rating_bands.loc[MODEL_TRAIN_INDEX].astype(str),
    }),
    pd.DataFrame({
        "split": "test",
        TARGET: y_model.loc[MODEL_TEST_INDEX],
        "rating_band": model_rating_bands.loc[MODEL_TEST_INDEX].astype(str),
    }),
])

split_summary = (
    split_check_df
    .groupby("split")
    .agg(
        books=(TARGET, "size"),
        mean_rating=(TARGET, "mean"),
        median_rating=(TARGET, "median"),
        min_rating=(TARGET, "min"),
        max_rating=(TARGET, "max"),
        rating_std=(TARGET, "std"),
    )
    .reindex(["train", "test"])
)
split_summary["share_of_data"] = split_summary["books"] / len(model_source_df)

split_summary.round(3)

#### Check split balance

Comparing the percentage of each rating band in train and test to verify the stratified split.

In [ ]:
# Check whether each rating band has a similar share in train and test.
rating_band_split_share = pd.crosstab(
    split_check_df["rating_band"],
    split_check_df["split"],
    normalize="columns",
).reindex(columns=["train", "test"])

(rating_band_split_share * 100).round(1)


### 3.4 Feature Description

The final modelling feature set keeps features that are simple to explain and available from the Goodreads-style book metadata. Some feature ideas use different column versions depending on the model: linear regression receives log-scaled versions of skewed count features, while tree-based models receive the raw numeric values.

| Feature / feature idea | Rationale | Model use |
|---|---|---|
| `ratings_count` / `log_ratings_count` | Rating volume shows how much support the average rating has. More ratings generally make the average rating more stable. | Log version for Linear Regression; raw version for tree models. |
| `text_reviews_count` / `log_text_reviews_count` | Written reviews are a stronger engagement signal than simple star ratings. | Log version for Linear Regression; raw version for tree models. |
| `review_to_rating_ratio` / `log_review_share` | Measures review intensity: among users who rated the book, how many wrote a text review. | Log version for Linear Regression; raw version for tree models. |
| `num_pages` / `log_num_pages` | Page count captures book length, which may relate to genre, format, or reader commitment. | Log version for Linear Regression; raw version for tree models. |
| `num_pages_missing` | Flags books where page count was missing or invalid before imputation, especially useful for audiobook-like records. | All models. |
| `book_age` | Older books have had more time to accumulate readers and may reflect classics or long-lived popularity. | All models. |
| `publication_decade` | Groups publication timing into broader historical periods without treating every exact year as a separate category. | One-hot encoded for sklearn models; categorical for CatBoost. |
| `language_code_grouped` | Keeps frequent language codes while grouping rare languages into `other_language`, avoiding a sparse feature. | One-hot encoded for sklearn models; categorical for CatBoost. |
| `n_authors` | Multiple authors can indicate anthologies, edited works, collaborations, or collections. | All models. |
| `first_author` | The main author can carry strong book-quality and audience information, but it is high-cardinality. | CatBoost only, because it can handle categorical features directly. |
| `first_author_frequency` / `log_first_author_frequency` | Counts how often the first author appears in the training data, giving a compact familiarity signal. | Log version for Linear Regression; raw version for tree models. |
| `first_author_target_mean` | Smoothed average rating for the first author, fitted on the training set only. | Sklearn tree models. |
| `first_author_rating_std` | Smoothed rating variability across the first author's books. | Sklearn tree models. |
| `publisher` | Publisher identity can reflect editorial segment, format, or catalogue type, but it is high-cardinality. | CatBoost only, because it can handle categorical features directly. |
| `publisher_frequency` / `log_publisher_frequency` | Counts how often the publisher appears in the training data, keeping a compact publisher signal. | Log version for Linear Regression; raw version for tree models. |
| `publisher_target_mean` | Smoothed average rating for the publisher, fitted on the training set only. | Sklearn tree models. |
| `publisher_rating_std` | Smoothed rating variability across the publisher's books. | Sklearn tree models. |
| `title_character_count` / `title_word_count` | Compact measures of title length and structure. | All models. |
| `is_audio` | Audiobook or audio-related records often behave differently for page count and reader engagement. | All models. |
| `is_collection` | Collections, boxed sets, complete works, and anthologies may have different rating behavior from single books. | All models. |
| `is_series` | Series entries may benefit from existing audience selection and continuity effects. | All models. |

### 3.5 Feature Engineering

This is the final feature engineering used by the models. The design follows the EDA findings above, but it is implemented here as a train-fitted pipeline:
- text fields are normalized again so future web-app inputs can be handled consistently;
- `publication_year`, `book_age`, and `publication_decade` are derived from the raw `publication_date` field;
- `num_pages = 0` is treated as missing, with a missing flag and train-fitted median imputation;
- author and publisher frequencies are learned from the training set only;
- author and publisher target means and rating variability are learned from the training set only, with unseen values falling back to global training statistics;
- common language-code groups are learned from the training set only;
- review engagement is captured as text reviews divided by total ratings;
- collection and series indicators are derived from the title text;
- one compact set of count, page, date, author, publisher, language, engagement, and title-structure features is then created consistently for train, test, and manual prediction inputs.

#### Set feature-engineering patterns

In [ ]:
# Constants used by the notebook feature engineering pipeline.
# They are repeated here instead of relying on EDA variables so this modelling section is readable on its own.
REFERENCE_YEAR = 2026
MIN_LANGUAGE_COUNT = 50
TARGET_ENCODING_SMOOTHING = 10

AUDIO_INDICATOR_PATTERN = r"""
audio(?:books?|go|text)?\b
|
\bcassettes?\b
|
\bcd\s*audio\b
|
\baudio\s*cd\b
|
\bmp3\b
|
\bsound\s+recording\b
|
\blistening\s+library\b
"""

SERIES_NUMBER_PATTERN = r"(?:\d+(?:\.\d+)?|[ivxlcdm]+|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve)"
SERIES_NUMBER_RANGE_PATTERN = rf"{SERIES_NUMBER_PATTERN}(?:\s*(?:[-\u2013\u2014,/&]|and)\s*{SERIES_NUMBER_PATTERN})*"

COLLECTION_TITLE_PATTERN = r"""
\b(?:box(?:ed)?\s+set|omnibus|antholog(?:y|ies)|collection)\b
|
\b(?:complete|collected|selected|essential|major)\s+
(?:works|novels|stories|short\s+stories|poems|poetry|plays|writings|letters|essays|tragedies|adventures|memoirs)\b
|
\b\d+\s*(?:book|volume)s?\s+(?:box(?:ed)?\s+set|set|collection)\b
|
\b(?:\d+\s+)?volumes?\s+set\b
"""

SERIES_TITLE_PATTERN = rf"""
\([^)]*(?:\#\s*{SERIES_NUMBER_RANGE_PATTERN}|\b(?:vol(?:ume)?s?\.?|books?|part|tome)\s+{SERIES_NUMBER_RANGE_PATTERN})[^)]*\)
|
(?<!\w)\#\s*{SERIES_NUMBER_RANGE_PATTERN}\b
|
\b(?:vol(?:ume)?s?\.?|part|tome)\s+{SERIES_NUMBER_RANGE_PATTERN}\b
"""

#### Input and normalization helpers

In [ ]:
# Helper functions for feature engineering.
def normalize_model_text(value: Any) -> str | None:
    """Normalize a single text value for stable feature keys."""
    if value is None or pd.isna(value):
        return None

    normalized_text = str(value).strip()
    if not normalized_text:
        return ""

    # The frequency mappings should not split the same author/publisher because of case or accents.
    normalized_text = normalized_text.lower()
    normalized_text = unicodedata.normalize("NFKD", normalized_text)
    normalized_text = "".join(
        character
        for character in normalized_text
        if not unicodedata.combining(character)
    )
    normalized_text = re.sub(r"\s+", " ", normalized_text)
    return normalized_text


def series_or_default(frame, column, default_value):
    """Return one column or a default-valued Series for manual/web-app style inputs."""
    if column in frame.columns:
        return frame[column]
    return pd.Series(default_value, index=frame.index)


def numeric_series(frame, column, default_value=0):
    """Coerce numeric inputs while keeping invalid values as missing."""
    return pd.to_numeric(series_or_default(frame, column, default_value), errors="coerce")


def normalized_text_series(frame, column, missing_value):
    """Normalize text columns and keep an explicit missing bucket."""
    values = series_or_default(frame, column, None).map(normalize_model_text)
    return values.fillna(missing_value).replace("", missing_value)


#### Smoothed target-encoding helper

These helpers learn smoothed category means and rating variability from the training target.

In [ ]:
def fit_smoothed_target_encoding(category_values, target_values):
    """Learn smoothed target means for one categorical feature."""
    encoding_data = pd.DataFrame({
        "category": category_values,
        "target": pd.to_numeric(target_values, errors="coerce"),
    }).dropna(subset=["target"])

    global_target_mean = encoding_data["target"].mean()
    category_summary = encoding_data.groupby("category", dropna=False)["target"].agg(["count", "mean"])

    smoothed_target_mean = (
        category_summary["count"] * category_summary["mean"]
        + TARGET_ENCODING_SMOOTHING * global_target_mean
    ) / (category_summary["count"] + TARGET_ENCODING_SMOOTHING)

    return smoothed_target_mean


def fit_smoothed_target_std_encoding(category_values, target_values):
    """Learn category rating variability, shrunk toward the global variance."""
    encoding_data = pd.DataFrame({
        "category": category_values,
        "target": pd.to_numeric(target_values, errors="coerce"),
    }).dropna(subset=["target"])

    global_variance = encoding_data["target"].var(ddof=0)
    grouped_target = encoding_data.groupby("category", dropna=False)["target"]
    category_count = grouped_target.count()
    category_variance = grouped_target.var(ddof=0).fillna(0)

    smoothed_variance = (
        category_count * category_variance
        + TARGET_ENCODING_SMOOTHING * global_variance
    ) / (category_count + TARGET_ENCODING_SMOOTHING)

    return np.sqrt(smoothed_variance.clip(lower=0))


#### Prepare consistent base features

Raw notebook rows, alternative datasets, and future web-form rows all pass through the same visible cleaning steps. Missing columns receive explicit defaults and invalid numeric or date values are coerced safely.

In [ ]:
def prepare_base_model_features(raw_data):
    """Prepare raw input columns before deriving model features."""
    frame = raw_data.copy()
    base_features = pd.DataFrame(index=frame.index)

    # Normalize text first, because later author, publisher, and title flags depend on stable strings.
    base_features["title"] = normalized_text_series(frame, "title", "missing_title")
    base_features["authors"] = normalized_text_series(frame, "authors", "missing_author")
    base_features["publisher"] = normalized_text_series(frame, "publisher", "missing_publisher")
    base_features["language_code"] = (
        series_or_default(frame, "language_code", "missing_language")
        .fillna("missing_language")
        .replace("", "missing_language")
        .astype(str)
    )

    # Counts are clipped at zero so a future bad input cannot create negative engagement features.
    base_features["ratings_count"] = numeric_series(frame, "ratings_count", 0).fillna(0).clip(lower=0)
    base_features["text_reviews_count"] = numeric_series(frame, "text_reviews_count", 0).fillna(0).clip(lower=0)
    base_features["num_pages_raw"] = numeric_series(frame, "num_pages", np.nan)

    publication_dates = pd.to_datetime(series_or_default(frame, "publication_date", pd.NaT), errors="coerce")
    base_features["publication_year"] = publication_dates.dt.year

    title_publisher_text = base_features["title"].fillna("") + " " + base_features["publisher"].fillna("")
    base_features["is_audio"] = title_publisher_text.str.contains(
        AUDIO_INDICATOR_PATTERN,
        regex=True,
        flags=re.IGNORECASE | re.VERBOSE,
        na=False,
    ).astype(int)

    # Keep the first author as a compact signal and count all listed authors separately.
    base_features["first_author"] = (
        base_features["authors"]
        .str.split("/", n=1)
        .str[0]
        .str.strip()
        .replace("", "missing_author")
        .fillna("missing_author")
    )
    base_features["n_authors"] = (
        base_features["authors"]
        .str.split("/")
        .str.len()
        .fillna(1)
        .astype(float)
    )

    return base_features

#### Fit train-only feature references

In [ ]:
# Fit the feature-engineering reference values on the training rows only.
def fit_feature_engineering_reference(raw_train_data, train_target):
    base_train = prepare_base_model_features(raw_train_data)
    train_target = pd.to_numeric(train_target.reindex(base_train.index), errors="coerce")

    # Medians are learned from train only, so the test set does not influence imputation.
    publication_year_median = base_train["publication_year"].median()
    if pd.isna(publication_year_median):
        publication_year_median = REFERENCE_YEAR

    valid_page_counts = base_train["num_pages_raw"].where(base_train["num_pages_raw"] > 0)
    global_page_median = valid_page_counts.median()
    if pd.isna(global_page_median):
        global_page_median = 300

    page_medians_by_audio = {}
    for audio_value in [0, 1]:
        group_median = valid_page_counts[base_train["is_audio"] == audio_value].median()
        if pd.isna(group_median):
            group_median = global_page_median
        page_medians_by_audio[audio_value] = float(group_median)

    # These mappings are fitted on train only to avoid leakage from the test set.
    language_counts = base_train["language_code"].value_counts(dropna=False)
    common_languages = set(language_counts[language_counts >= MIN_LANGUAGE_COUNT].index)
    global_target_mean = train_target.mean()
    global_target_std = train_target.std(ddof=0)

    return {
        "publication_year_median": float(publication_year_median),
        "global_page_median": float(global_page_median),
        "page_medians_by_audio": page_medians_by_audio,
        "first_author_frequency_counts": base_train["first_author"].value_counts(dropna=False),
        "publisher_frequency_counts": base_train["publisher"].value_counts(dropna=False),
        "target_global_mean": float(global_target_mean),
        "target_global_std": float(global_target_std),
        "first_author_target_mean": fit_smoothed_target_encoding(base_train["first_author"], train_target),
        "publisher_target_mean": fit_smoothed_target_encoding(base_train["publisher"], train_target),
        "first_author_rating_std": fit_smoothed_target_std_encoding(base_train["first_author"], train_target),
        "publisher_rating_std": fit_smoothed_target_std_encoding(base_train["publisher"], train_target),
        "common_languages": common_languages,
    }

#### Build core model features

These features use the normalized raw values and train-fitted date and page-count references.


In [ ]:
def build_core_model_features(base_features, feature_reference):
    """Create date, engagement, page-count, and author-count features."""
    engineered = pd.DataFrame(index=base_features.index)

    engineered["first_author"] = base_features["first_author"]
    engineered["publisher"] = base_features["publisher"]

    # Date features use the train median when publication_date is missing or invalid.
    publication_year = base_features["publication_year"].fillna(feature_reference["publication_year_median"])
    engineered["book_age"] = (REFERENCE_YEAR - publication_year).clip(lower=0)
    engineered["publication_decade"] = ((publication_year.astype(int) // 10) * 10).astype(str) + "s"

    # Engagement features keep rating volume and the share of ratings that produced text reviews.
    engineered["ratings_count"] = base_features["ratings_count"]
    engineered["text_reviews_count"] = base_features["text_reviews_count"]
    ratings_for_ratio = engineered["ratings_count"].replace(0, np.nan)
    engineered["review_to_rating_ratio"] = (engineered["text_reviews_count"] / ratings_for_ratio).fillna(0)
    engineered["log_review_share"] = np.log1p(engineered["review_to_rating_ratio"].clip(lower=0))
    engineered["is_audio"] = base_features["is_audio"].astype(int)

    # Page counts of zero are treated as missing, then imputed by audiobook/non-audiobook median.
    engineered["num_pages_missing"] = (base_features["num_pages_raw"].isna() | base_features["num_pages_raw"].le(0)).astype(int)
    engineered["num_pages"] = base_features["num_pages_raw"].where(base_features["num_pages_raw"] > 0).astype(float)
    for audio_value, median_page_count in feature_reference["page_medians_by_audio"].items():
        group_mask = engineered["is_audio"].eq(audio_value)
        engineered.loc[group_mask, "num_pages"] = engineered.loc[group_mask, "num_pages"].fillna(median_page_count)
    engineered["num_pages"] = engineered["num_pages"].fillna(feature_reference["global_page_median"])

    engineered["n_authors"] = base_features["n_authors"].fillna(1).clip(lower=1)
    return engineered

#### Add frequency and target-statistic encodings

The mappings come only from the fitted reference. Unseen categories receive zero frequency and global training-target statistics.

In [ ]:
def add_reference_encoded_features(engineered, base_features, feature_reference):
    """Add train-fitted frequency, target statistics, and language features."""
    engineered["first_author_frequency"] = (
        base_features["first_author"]
        .map(feature_reference["first_author_frequency_counts"])
        .fillna(0)
        .astype(int)
    )
    engineered["publisher_frequency"] = (
        base_features["publisher"]
        .map(feature_reference["publisher_frequency_counts"])
        .fillna(0)
        .astype(int)
    )

    engineered["first_author_target_mean"] = (
        base_features["first_author"]
        .map(feature_reference["first_author_target_mean"])
        .fillna(feature_reference["target_global_mean"])
        .astype(float)
    )
    engineered["publisher_target_mean"] = (
        base_features["publisher"]
        .map(feature_reference["publisher_target_mean"])
        .fillna(feature_reference["target_global_mean"])
        .astype(float)
    )
    engineered["first_author_rating_std"] = (
        base_features["first_author"]
        .map(feature_reference["first_author_rating_std"])
        .fillna(feature_reference["target_global_std"])
        .astype(float)
    )
    engineered["publisher_rating_std"] = (
        base_features["publisher"]
        .map(feature_reference["publisher_rating_std"])
        .fillna(feature_reference["target_global_std"])
        .astype(float)
    )

    engineered["language_code_grouped"] = base_features["language_code"].where(
        base_features["language_code"].isin(feature_reference["common_languages"]),
        "other_language",
    )
    return engineered


#### Add title features and log transformations

In [ ]:
def add_title_and_log_features(engineered, base_features):
    """Add title-derived counts and flags plus log-transformed features."""
    title_text = base_features["title"].where(base_features["title"].ne("missing_title"), "")
    engineered["title_character_count"] = title_text.str.len().fillna(0).astype(int)
    engineered["title_word_count"] = title_text.str.split().str.len().fillna(0).astype(int)
    engineered["is_collection"] = title_text.str.contains(
        COLLECTION_TITLE_PATTERN,
        regex=True,
        flags=re.IGNORECASE | re.VERBOSE,
        na=False,
    ).astype(int)
    engineered["is_series"] = title_text.str.contains(
        SERIES_TITLE_PATTERN,
        regex=True,
        flags=re.IGNORECASE | re.VERBOSE,
        na=False,
    ).astype(int)

    engineered["log_ratings_count"] = np.log1p(engineered["ratings_count"])
    engineered["log_text_reviews_count"] = np.log1p(engineered["text_reviews_count"])
    engineered["log_num_pages"] = np.log1p(engineered["num_pages"])
    engineered["log_first_author_frequency"] = np.log1p(engineered["first_author_frequency"])
    engineered["log_publisher_frequency"] = np.log1p(engineered["publisher_frequency"])
    return engineered


#### Apply the complete feature-engineering sequence

This small orchestrator makes the order of the three transformation stages explicit for training, testing, alternative datasets, and future web inputs.

In [ ]:
def apply_feature_engineering(raw_data, feature_reference):
    """Apply the complete fitted feature-engineering sequence."""
    base_features = prepare_base_model_features(raw_data)
    engineered = build_core_model_features(base_features, feature_reference)
    engineered = add_reference_encoded_features(engineered, base_features, feature_reference)
    engineered = add_title_and_log_features(engineered, base_features)
    return engineered


#### Create engineered train and test tables

Fitting feature engineering on raw training rows and applying the learned references separately to train and test.

In [ ]:
# Fit feature engineering on train only, then apply it to train and test.
feature_engineering_reference = fit_feature_engineering_reference(
    X_model_raw.loc[MODEL_TRAIN_INDEX],
    y_model.loc[MODEL_TRAIN_INDEX],
)

X_train_engineered = apply_feature_engineering(
    X_model_raw.loc[MODEL_TRAIN_INDEX],
    feature_engineering_reference,
)
X_test_engineered = apply_feature_engineering(
    X_model_raw.loc[MODEL_TEST_INDEX],
    feature_engineering_reference,
)

feature_engineering_summary = pd.DataFrame([
    {"metric": "train_rows", "value": len(X_train_engineered)},
    {"metric": "test_rows", "value": len(X_test_engineered)},
    {"metric": "engineered_columns", "value": X_train_engineered.shape[1]},
    {"metric": "common_languages_learned", "value": len(feature_engineering_reference["common_languages"])},
    {"metric": "target_encoding_smoothing", "value": TARGET_ENCODING_SMOOTHING},
    {"metric": "author_target_mappings", "value": len(feature_engineering_reference["first_author_target_mean"])},
    {"metric": "publisher_target_mappings", "value": len(feature_engineering_reference["publisher_target_mean"])},
    {"metric": "author_std_mappings", "value": len(feature_engineering_reference["first_author_rating_std"])},
    {"metric": "publisher_std_mappings", "value": len(feature_engineering_reference["publisher_rating_std"])},
])

feature_engineering_summary

#### Preview engineered features

In [ ]:
# Preview representative engineered columns without producing a very wide table.
engineered_preview_columns = [
    "book_age",
    "num_pages",
    "num_pages_missing",
    "ratings_count",
    "review_to_rating_ratio",
    "first_author_frequency",
    "first_author_target_mean",
    "publisher_frequency",
    "language_code_grouped",
    "is_series",
]

X_train_engineered[engineered_preview_columns].head()

### 3.6 Feature Selection

The feature lists combine shared features with small family-specific groups: linear regression receives log-transformed skewed values, sklearn tree models receive raw values and compact target statistics, and CatBoost receives raw categorical metadata.


#### Select features by model family

In [ ]:
# Shared structural features are used by every model family.
SHARED_NUMERIC_FEATURES = [
    "book_age",
    "n_authors",
    "num_pages_missing",
    "title_character_count",
    "title_word_count",
    "is_audio",
    "is_collection",
    "is_series",
]

LOW_CARDINALITY_CATEGORICAL_FEATURES = [
    "language_code_grouped",
    "publication_decade",
]

# Linear regression uses log transformations for skewed counts.
LINEAR_NUMERIC_FEATURES = [
    "log_ratings_count",
    "log_text_reviews_count",
    "log_review_share",
    "log_num_pages",
    "log_first_author_frequency",
    "log_publisher_frequency",
    *SHARED_NUMERIC_FEATURES,
]

# Sklearn tree models share raw values and compact author/publisher target statistics.
TREE_NUMERIC_FEATURES = [
    "ratings_count",
    "text_reviews_count",
    "review_to_rating_ratio",
    "num_pages",
    "first_author_frequency",
    "publisher_frequency",
    "first_author_target_mean",
    "publisher_target_mean",
    "first_author_rating_std",
    "publisher_rating_std",
    *SHARED_NUMERIC_FEATURES,
]

# CatBoost uses raw categorical identities, so it only needs non-target numeric aggregates.
CATBOOST_NUMERIC_FEATURES = [
    "ratings_count",
    "text_reviews_count",
    "review_to_rating_ratio",
    "num_pages",
    "first_author_frequency",
    "publisher_frequency",
    *SHARED_NUMERIC_FEATURES,
]
CATBOOST_CATEGORICAL_FEATURES = [
    "first_author",
    "publisher",
    *LOW_CARDINALITY_CATEGORICAL_FEATURES,
]

def make_model_spec(
    numeric_features,
    categorical_features,
    reported_parameters,
    *,
    fit_strategy="sklearn_pipeline",
    scale_numeric=False,
):
    """Build one compact model-family feature specification."""
    return {
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "fit_strategy": fit_strategy,
        "scale_numeric": scale_numeric,
        "reported_parameters": reported_parameters,
    }


# Each spec contains everything required by the shared training helper.
MODEL_SPECS = {
    "Linear regression": make_model_spec(
        LINEAR_NUMERIC_FEATURES, LOW_CARDINALITY_CATEGORICAL_FEATURES, [], scale_numeric=True
    ),
    "HistGradientBoosting": make_model_spec(
        TREE_NUMERIC_FEATURES,
        LOW_CARDINALITY_CATEGORICAL_FEATURES,
        ["max_iter", "learning_rate", "max_leaf_nodes", "l2_regularization", "min_samples_leaf"],
    ),
    "XGBoost": make_model_spec(
        TREE_NUMERIC_FEATURES,
        LOW_CARDINALITY_CATEGORICAL_FEATURES,
        ["n_estimators", "learning_rate", "max_depth", "min_child_weight"],
    ),
    "Random Forest": make_model_spec(
        TREE_NUMERIC_FEATURES,
        LOW_CARDINALITY_CATEGORICAL_FEATURES,
        ["n_estimators", "max_features", "min_samples_leaf"],
    ),
    "Extra Trees": make_model_spec(
        TREE_NUMERIC_FEATURES,
        LOW_CARDINALITY_CATEGORICAL_FEATURES,
        ["n_estimators", "max_features", "min_samples_leaf"],
    ),
    "CatBoost": make_model_spec(
        CATBOOST_NUMERIC_FEATURES,
        CATBOOST_CATEGORICAL_FEATURES,
        ["iterations", "learning_rate", "depth", "l2_leaf_reg"],
        fit_strategy="native_categorical",
    ),
}

model_feature_overview = pd.DataFrame([
    {
        "model": model_name,
        "fit_strategy": model_spec["fit_strategy"],
        "numeric_features": len(model_spec["numeric_features"]),
        "categorical_features": len(model_spec["categorical_features"]),
        "total_input_columns": (
            len(model_spec["numeric_features"])
            + len(model_spec["categorical_features"])
        ),
    }
    for model_name, model_spec in MODEL_SPECS.items()
])

model_feature_overview

#### Validate selected feature names

Confirming that every configured feature exists before fitting any model.

In [ ]:
# Confirm that every selected feature exists in the engineered training table.
selected_feature_names = sorted({
    feature_name
    for model_spec in MODEL_SPECS.values()
    for feature_name in (
        model_spec["numeric_features"]
        + model_spec["categorical_features"]
    )
})

missing_selected_features = [
    feature_name
    for feature_name in selected_feature_names
    if feature_name not in X_train_engineered.columns
]

pd.DataFrame({
    "selected_features": [len(selected_feature_names)],
    "missing_selected_features": [missing_selected_features],
})


### 3.7 Active Model Training

#### Defining active model configurations

In [ ]:
# Active model configurations used for normal notebook runs.
active_model_configs = [
    {
        "model": "Linear regression",
        "estimator": LinearRegression(),
    },
    {
        "model": "HistGradientBoosting", # tuning done
        "estimator": HistGradientBoostingRegressor(
            max_iter=75,
            learning_rate=0.13,
            max_leaf_nodes=63,
            l2_regularization=1,
            min_samples_leaf=20,
            early_stopping=False,
            random_state=RANDOM_STATE,
        ),
    },
    {
        "model": "XGBoost", # tuning done
        "estimator": XGBRegressor(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=6,
            min_child_weight=6,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=5.0,
            objective="reg:squarederror",
            eval_metric="mae",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    },
    {
        "model": "Random Forest", # tuning done
        "estimator": RandomForestRegressor(
            n_estimators=200,
            max_features=0.8,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    },
    {
        "model": "Extra Trees", # tuning done
        "estimator": ExtraTreesRegressor(
            n_estimators=200,
            max_features=0.7,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    },
    {
        "model": "CatBoost",
        "estimator": CatBoostRegressor(
            iterations=400,
            learning_rate=0.09,
            depth=5,
            l2_leaf_reg=3,
            loss_function="RMSE",
            eval_metric="MAE",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
        ),
    },
]

#### Training helpers

In [ ]:
# Training helpers used by the active and tuning model runs.
def make_one_hot_encoder():
    """Create a OneHotEncoder that works across recent sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def prepare_feature_matrix(engineered_data, numeric_features, categorical_features):
    """Select the engineered columns required by one model family."""
    feature_names = numeric_features + categorical_features
    X = engineered_data[feature_names].copy()

    bool_columns = X.select_dtypes(include="bool").columns
    X[bool_columns] = X[bool_columns].astype(int)

    for feature_name in numeric_features:
        X[feature_name] = pd.to_numeric(X[feature_name], errors="coerce")

    for feature_name in categorical_features:
        X[feature_name] = X[feature_name].astype("object").where(X[feature_name].notna(), "Missing").astype(str)

    return X


def build_sklearn_pipeline(model, numeric_features, categorical_features, scale_numeric=False):
    """Build preprocessing plus any sklearn-compatible regressor."""
    if not numeric_features and not categorical_features:
        raise ValueError("At least one numeric or categorical feature is required.")

    transformers = []

    if numeric_features:
        numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
        if scale_numeric:
            numeric_steps.append(("scaler", StandardScaler()))
        transformers.append(("num", Pipeline(steps=numeric_steps), numeric_features))

    if categorical_features:
        categorical_transformer = Pipeline(steps=[
            ("onehot", make_one_hot_encoder()),
        ])
        transformers.append(("cat", categorical_transformer, categorical_features))

    return Pipeline(steps=[
        ("preprocessor", ColumnTransformer(transformers=transformers)),
        ("model", clone(model)),
    ])


def model_parameters_for_result(model_name, estimator):
    """Extract the model settings shown in result tables and model lookups."""
    estimator_params = estimator.get_params()
    return {
        parameter_name: estimator_params[parameter_name]
        for parameter_name in MODEL_SPECS[model_name]["reported_parameters"]
        if parameter_name in estimator_params
    }


def format_parameters(parameters):
    """Create a compact readable label for notebook tables and model lookup keys, one per line."""
    if not parameters:
        return "default"
    return ", ".join(f"{key}={value}" for key, value in parameters.items())


def regression_metrics(y_true, y_pred):
    """Compute the regression metrics used in the project report."""
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


#### Evaluate model configurations

Fitting each active configuration on the fixed training rows and collecting comparable test metrics.

In [ ]:
def fit_model_configuration(config, engineered_data, target_values):
    """Fit one configured model on any engineered dataset."""
    model_name = config["model"]
    model_spec = MODEL_SPECS[model_name]
    numeric_features = model_spec["numeric_features"]
    categorical_features = model_spec["categorical_features"]
    X = prepare_feature_matrix(engineered_data, numeric_features, categorical_features)

    if model_spec["fit_strategy"] == "native_categorical":
        fitted_model = clone(config["estimator"])
        fitted_model.fit(X, target_values, cat_features=categorical_features)
    elif model_spec["fit_strategy"] == "sklearn_pipeline":
        fitted_model = build_sklearn_pipeline(
            config["estimator"],
            numeric_features,
            categorical_features,
            scale_numeric=model_spec["scale_numeric"],
        )
        fitted_model.fit(X, target_values)
    else:
        raise ValueError(f"Unknown fit strategy for {model_name!r}: {model_spec['fit_strategy']}")

    model_parameters = model_parameters_for_result(model_name, config["estimator"])
    return {
        "model": fitted_model,
        "model_name": model_name,
        "parameters": format_parameters(model_parameters),
        "numeric_features": list(numeric_features),
        "categorical_features": list(categorical_features),
    }


def fit_and_evaluate_model(config):
    """Fit one model configuration and return metrics plus the fitted model."""
    model_name = config["model"]
    y_train = y_model.loc[MODEL_TRAIN_INDEX]
    y_test = y_model.loc[MODEL_TEST_INDEX]

    fitted_record = fit_model_configuration(config, X_train_engineered, y_train)
    X_test = prepare_feature_matrix(
        X_test_engineered,
        fitted_record["numeric_features"],
        fitted_record["categorical_features"],
    )
    y_pred = np.clip(fitted_record["model"].predict(X_test), 0, 5)
    metrics = regression_metrics(y_test, y_pred)

    result_row = {
        "model": model_name,
        "n_numeric_features": len(fitted_record["numeric_features"]),
        "n_categorical_features": len(fitted_record["categorical_features"]),
        "parameters": fitted_record["parameters"],
    }
    result_row.update(metrics)

    fitted_key = (model_name, fitted_record["parameters"])
    return result_row, fitted_key, fitted_record


def evaluate_model_configs(configs):
    """Evaluate several configurations and keep fitted models for prediction checks."""
    rows = []
    fitted_models = {}

    for config in configs:
        row, fitted_key, fitted_record = fit_and_evaluate_model(config)
        rows.append(row)
        fitted_models[fitted_key] = fitted_record

    return pd.DataFrame(rows), fitted_models


def fit_and_evaluate_constant_baseline():
    """Evaluate a mean DummyRegressor without using the feature pipeline."""
    y_train = y_model.loc[MODEL_TRAIN_INDEX]
    y_test = y_model.loc[MODEL_TEST_INDEX]
    X_train_constant = np.ones((len(y_train), 1))
    X_test_constant = np.ones((len(y_test), 1))

    baseline_model = DummyRegressor(strategy="mean")
    baseline_model.fit(X_train_constant, y_train)
    baseline_predictions = np.clip(baseline_model.predict(X_test_constant), 0, 5)
    baseline_metrics = regression_metrics(y_test, baseline_predictions)

    baseline_result = {
        "model": "Constant mean baseline",
        "n_numeric_features": 0,
        "n_categorical_features": 0,
        "parameters": "strategy=mean",
    }
    baseline_result.update(baseline_metrics)
    return baseline_result


#### Define prediction helper

In [ ]:
# The fitted reference is explicit so the same helper works for evaluation and exported models.
def predict_from_raw_input(fitted_record, raw_data, feature_reference):
    engineered_data = apply_feature_engineering(raw_data, feature_reference)
    X_input = prepare_feature_matrix(
        engineered_data,
        fitted_record["numeric_features"],
        fitted_record["categorical_features"],
    )
    return np.clip(fitted_record["model"].predict(X_input), 0, 5)


#### Run active model comparison

The constant mean baseline predicts the training-set average for every book. The median is theoretically optimal for MAE, but the mean provides the more intuitive "predict the average" reference point.

In [ ]:
# Evaluate the active configurations and the separate mean baseline on the fixed split.
model_results, fitted_models = evaluate_model_configs(active_model_configs)
constant_baseline_result = fit_and_evaluate_constant_baseline()
model_results = pd.concat(
    [model_results, pd.DataFrame([constant_baseline_result])],
    ignore_index=True,
)

model_results.sort_values("MAE").round({"MAE": 3, "RMSE": 3, "R2": 3})


### 3.8 Model tuning

The optional tuning grid compares parameter combinations for the selected model using training data. The held-out test set remains reserved for final evaluation.

#### Prepare the tuning grid

In [ ]:
RUN_TUNING = False

# Configure exactly one model family for each tuning run.
tuning_model_config = {
    "model": "CatBoost",
    "estimator": CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="MAE",
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
    ),
    "param_grid": {
        "iterations": [400],
        "learning_rate": [0.09],
        "depth": [5, 6, 7],
        "l2_leaf_reg": [3.0, 5.0, 7.0],
    },
}

# HistGradientBoosting.
# tuning_model_config = {
#     "model": "HistGradientBoosting",
#     "estimator": HistGradientBoostingRegressor(
#         random_state=RANDOM_STATE,
#         early_stopping=False,
#     ),
#     "param_grid": {
#         "max_iter": [75],
#         "learning_rate": [0.13],
#         "max_leaf_nodes": [31, 63, 127],
#         "min_samples_leaf": [10, 20, 40],
#         "l2_regularization": [0.5, 1.0, 2.0],
#     },
# }

# XGBoost.
# tuning_model_config = {
#     "model": "XGBoost",
#     "estimator": XGBRegressor(
#         subsample=0.8,
#         colsample_bytree=0.8,
#         reg_lambda=5.0,
#         objective="reg:squarederror",
#         eval_metric="mae",
#         tree_method="hist",
#         random_state=RANDOM_STATE,
#         n_jobs=-1,
#     ),
#     "param_grid": {
#         "n_estimators": [400,600],
#         "learning_rate": [0.05,0.06],
#         "max_depth": [6,7],
#         "min_child_weight": [6,7],
#     },
# }

# Random Forest.
# tuning_model_config = {
#     "model": "Random Forest",
#     "estimator": RandomForestRegressor(
#         random_state=RANDOM_STATE,
#         n_jobs=-1,
#     ),
#     "param_grid": {
#         "n_estimators": [100, 200, 300],
#         "max_features": [0.6, 0.7, 0.8],
#         "min_samples_leaf": [1, 2, 3],
#     },
# }

# Extra Trees.
# tuning_model_config = {
#     "model": "Extra Trees",
#     "estimator": ExtraTreesRegressor(
#         random_state=RANDOM_STATE,
#         n_jobs=-1,
#     ),
#     "param_grid": {
#         "n_estimators": [100, 200, 300],
#         "max_features": [0.6, 0.7, 0.8],
#         "min_samples_leaf": [1, 2, 3],
#     },
# }


def expand_param_grid(param_grid):
    """Return every parameter combination from a grid dictionary."""
    parameter_names = list(param_grid.keys())
    parameter_values = [param_grid[name] for name in parameter_names]
    return [
        dict(zip(parameter_names, values))
        for values in product(*parameter_values)
    ]


if tuning_model_config["model"] not in MODEL_SPECS:
    raise ValueError(f"Unknown tuning model: {tuning_model_config['model']!r}")

tuning_parameter_sets = (
    expand_param_grid(tuning_model_config["param_grid"])
    if RUN_TUNING
    else []
)
tuning_grid_summary = pd.DataFrame([{
    "model": tuning_model_config["model"],
    "grid_configurations": len(tuning_parameter_sets),
    "status": "ready" if RUN_TUNING else "skipped",
}])

tuning_grid_summary

#### Run tuning when enabled

In [ ]:
# Build candidate configurations for the one selected model only.
if RUN_TUNING:
    tuning_model_configs = [
        {
            "model": tuning_model_config["model"],
            "estimator": clone(tuning_model_config["estimator"]).set_params(**params),
        }
        for params in tuning_parameter_sets
    ]
    tuning_results, fitted_tuned_models = evaluate_model_configs(tuning_model_configs)

    tuning_results_ranked = tuning_results.sort_values(
        by=["MAE", "RMSE", "R2"],
        ascending=[True, True, False],
    ).reset_index(drop=True)
    best_tuning_results = tuning_results_ranked.head(1).copy()
else:
    tuning_model_configs = []
    tuning_results = pd.DataFrame(columns=[
        "model",
        "n_numeric_features",
        "n_categorical_features",
        "parameters",
        "MAE",
        "RMSE",
        "R2",
    ])
    fitted_tuned_models = {}
    tuning_results_ranked = tuning_results.copy()
    best_tuning_results = tuning_results.copy()
    print(f"Skipping tuning for {tuning_model_config['model']} because RUN_TUNING is False.")

best_tuning_results.round({"MAE": 3, "RMSE": 3, "R2": 3})


#### Prepare tuning summary

Keeping the best tuning candidates when tuning has run, otherwise returning an empty summary.

In [ ]:
# Prepare the top candidates for the selected model only.
if RUN_TUNING and not tuning_results_ranked.empty:
    top_tuning_results = tuning_results_ranked.head(20).reset_index(drop=True)
else:
    top_tuning_results = pd.DataFrame(columns=["model", "parameters", "MAE", "RMSE", "R2"])
    print("No tuning candidates to display because RUN_TUNING is False.")


#### Display tuning candidates

In [ ]:
top_tuning_results[["model", "parameters", "MAE", "RMSE", "R2"]].head(10)


### 3.9 Final evaluation

Ranking the active configurations from Section 3.7 on the same test rows. Any accepted tuning result must first be copied into the active configuration and rerun through the normal comparison.

#### Rank final active models

In [ ]:
# Rank the active model configurations by MAE, with RMSE and R2 available for context.
final_model_results = (
    model_results.round({"MAE": 3, "RMSE": 3, "R2": 3})
    .sort_values(["MAE", "RMSE", "R2"], ascending=[True, True, False])
    .reset_index(drop=True)
)

final_model_results


#### Visualize the final model comparison

Showing MAE and RMSE together separates typical absolute error from the stronger penalty RMSE applies to larger misses. Lower values are better for both metrics.

In [ ]:
model_comparison_plot = final_model_results.sort_values("MAE", ascending=False).reset_index(drop=True)
best_predictive_model_name = (
    final_model_results[final_model_results["model"].isin(MODEL_SPECS)]
    .sort_values(["MAE", "RMSE", "R2"], ascending=[True, True, False])
    .iloc[0]["model"]
)
y_positions = np.arange(len(model_comparison_plot))

fig, ax = plt.subplots(figsize=(10.5, 6.4))
for y_position, row in model_comparison_plot.iterrows():
    line_color = COLOR_PRIMARY if row["model"] == best_predictive_model_name else COLOR_LIGHT
    line_width = 5 if row["model"] == best_predictive_model_name else 3
    ax.hlines(
        y_position,
        row["MAE"],
        row["RMSE"],
        color=line_color,
        linewidth=line_width,
        zorder=1,
    )

mae_colors = [
    COLOR_PRIMARY if model == best_predictive_model_name else COLOR_BLUE
    for model in model_comparison_plot["model"]
]
ax.scatter(
    model_comparison_plot["MAE"],
    y_positions,
    s=110,
    color=mae_colors,
    edgecolor="white",
    linewidth=1,
    label="MAE",
    zorder=3,
)
ax.scatter(
    model_comparison_plot["RMSE"],
    y_positions,
    s=110,
    color=COLOR_GOLD,
    edgecolor="white",
    linewidth=1,
    marker="D",
    label="RMSE",
    zorder=3,
)
for y_position, row in model_comparison_plot.iterrows():
    ax.text(
        row["MAE"], y_position + 0.16, f"{row['MAE']:.3f}",
        ha="center", va="bottom", fontsize=8.5,
    )
    ax.text(
        row["RMSE"], y_position - 0.16, f"{row['RMSE']:.3f}",
        ha="center", va="top", fontsize=8.5,
    )
ax.set(
    title="Non-linear ensembles substantially outperform the mean baseline",
    xlabel="Holdout error in rating points (lower is better)",
    ylabel="Model",
    yticks=y_positions,
    yticklabels=model_comparison_plot["model"],
)
ax.legend(frameon=False, ncol=2, loc="upper right", bbox_to_anchor=(1, 1.10))
ax.grid(axis="y", visible=False)

plt.tight_layout()
plt.show()

The tree-based ensembles form a tight leading group and clearly improve on linear regression and the constant baseline. The top MAE values differ by only a few thousandths of a rating point, so the selected winner is narrow rather than dominant.

#### Select the best fitted model

Storing the best active model for diagnostics and later export.

In [ ]:
selected_model_name = None  # Set to a configured model name, or keep None for the best model.
# The baseline is comparison-only and can never become the selected/exported model.
selectable_model_results = final_model_results[
    final_model_results["model"].isin(MODEL_SPECS)
]
if selected_model_name is None:
    selected_model_name = selectable_model_results["model"].iloc[0]

if selected_model_name not in MODEL_SPECS:
    raise ValueError(
        f"Selected model {selected_model_name!r} is not a configured predictive model."
    )

selected_result_rows = selectable_model_results[
    selectable_model_results["model"] == selected_model_name
]
if len(selected_result_rows) != 1:
    raise ValueError(
        f"Expected one final result for {selected_model_name!r}; "
        f"found {len(selected_result_rows)}."
    )
best_final_model = selected_result_rows.iloc[0]

best_model_key = (best_final_model["model"], best_final_model["parameters"])
best_fitted_record = fitted_models[best_model_key]

print(
    f"Best final model: {best_final_model['model']} "
    f"with MAE = {best_final_model['MAE']:.3f}"
)

### 3.10 Feature Impact Analysis

This optional analysis answers two related questions:

1. How well does each feature perform by itself compared with the constant mean baseline?
2. How does test MAE change as features are added in that standalone ranking order?

Every score comes from a fresh clone of the selected model using its current estimator settings, preprocessing strategy, and fitted-record feature lists.

#### Set analysis controls

Feature-impact analysis refits the selected model many times, so it can be switched off independently of the main model comparison.

In [ ]:
# Guard to avoid running the analysis multiple times.
RUN_FEATURE_IMPACT_ANALYSIS = True

# The final cumulative fit and a fresh full-feature fit should have the same MAE.
FEATURE_IMPACT_MAE_TOLERANCE = 1e-9

#### Select the active model configuration

Use the configuration that produced the selected model in the active model comparison above.

In [ ]:
selected_model_spec = MODEL_SPECS[selected_model_name]

selected_model_configs = [
    config
    for config in active_model_configs
    if config["model"] == selected_model_name
]

if len(selected_model_configs) != 1:
    raise ValueError(
        f"Expected exactly one active configuration for {selected_model_name!r}; "
        f"found {len(selected_model_configs)}."
    )

selected_model_config = selected_model_configs[0]

#### Read and validate the selected features

The fitted record is the source of truth because it contains the exact numeric and categorical feature lists evaluated above. The checks below catch stale notebook state before the expensive refits begin.

In [ ]:
selected_numeric_features = list(best_fitted_record["numeric_features"])
selected_categorical_features = list(best_fitted_record["categorical_features"])
selected_feature_order = selected_numeric_features + selected_categorical_features

if not selected_feature_order:
    raise ValueError(f"Selected model {selected_model_name!r} has no configured features.")

if len(selected_feature_order) != len(set(selected_feature_order)):
    raise ValueError(f"Selected model {selected_model_name!r} contains duplicate features.")

features_match_model_spec = (
    selected_numeric_features == list(selected_model_spec["numeric_features"])
    and selected_categorical_features == list(selected_model_spec["categorical_features"])
)
if not features_match_model_spec:
    raise ValueError(
        "The selected fitted record does not match the current MODEL_SPECS feature lists. "
        "Rerun the active model comparison before feature-impact analysis."
    )

#### Prepare a feature subset

Validate the requested names, preserve the selected model's feature order, and build matching train and test matrices.

In [ ]:
selected_feature_set = set(selected_feature_order)


def prepare_feature_subset(feature_names):
    """Validate a feature subset and prepare its train and test matrices."""
    requested_features = list(feature_names)

    # Catch invalid requests before starting an expensive model fit.
    if not requested_features:
        raise ValueError("Feature-impact evaluation requires at least one feature.")
    if len(requested_features) != len(set(requested_features)):
        raise ValueError("Feature-impact subsets cannot contain duplicate feature names.")

    unknown_features = sorted(set(requested_features) - selected_feature_set)
    if unknown_features:
        raise ValueError(
            f"Features are not used by {selected_model_name!r}: {unknown_features}"
        )

    # Preserve the feature order used by the selected model.
    requested_feature_set = set(requested_features)
    numeric_features = [
        feature
        for feature in selected_numeric_features
        if feature in requested_feature_set
    ]
    categorical_features = [
        feature
        for feature in selected_categorical_features
        if feature in requested_feature_set
    ]

    X_train_subset = prepare_feature_matrix(
        X_train_engineered,
        numeric_features,
        categorical_features,
    )
    X_test_subset = prepare_feature_matrix(
        X_test_engineered,
        numeric_features,
        categorical_features,
    )

    return (
        X_train_subset,
        X_test_subset,
        numeric_features,
        categorical_features,
    )

#### Fit and score a feature subset

Fit a fresh clone of the selected model with the same strategy used in the main model comparison, then calculate hold-out metrics.

In [ ]:
feature_impact_y_train = y_model.loc[MODEL_TRAIN_INDEX]
feature_impact_y_test = y_model.loc[MODEL_TEST_INDEX]


def fit_and_evaluate_feature_subset(feature_names):
    """Fit the selected model on a feature subset and return hold-out metrics."""
    (
        X_train_subset,
        X_test_subset,
        numeric_features,
        categorical_features,
    ) = prepare_feature_subset(feature_names)

    estimator = clone(selected_model_config["estimator"])
    fit_strategy = selected_model_spec["fit_strategy"]

    if fit_strategy == "sklearn_pipeline":
        fitted_model = build_sklearn_pipeline(
            estimator,
            numeric_features,
            categorical_features,
            scale_numeric=selected_model_spec["scale_numeric"],
        )
        fitted_model.fit(X_train_subset, feature_impact_y_train)
    elif fit_strategy == "native_categorical":
        fitted_model = estimator
        fitted_model.fit(
            X_train_subset,
            feature_impact_y_train,
            cat_features=categorical_features,
        )
    else:
        raise ValueError(
            f"Unknown fit strategy for {selected_model_name!r}: {fit_strategy!r}"
        )

    predictions = np.clip(fitted_model.predict(X_test_subset), 0, 5)
    return regression_metrics(feature_impact_y_test, predictions)

#### Initialize the analysis

Reset the optional outputs so rerunning the notebook cannot leave results from an older model selection in memory.

In [ ]:
standalone_feature_results = None
cumulative_feature_results = None
feature_impact_consistency = None
feature_impact_waterfall = None

if RUN_FEATURE_IMPACT_ANALYSIS:
    baseline_mae = float(constant_baseline_result["MAE"])
    total_fits = 2 * len(selected_feature_order) + 1
    print(
        f"Running {total_fits} feature-impact fits for {selected_model_name} "
        "with its current active estimator parameters."
    )
else:
    print(
        "Feature-impact analysis is disabled. "
        "Set RUN_FEATURE_IMPACT_ANALYSIS = True to run it manually."
    )

#### Rank features by standalone performance

Fit one model per feature. A positive `standalone_MAE_gain` means that the feature beats the constant mean baseline when used alone.

In [ ]:
if RUN_FEATURE_IMPACT_ANALYSIS:
    standalone_rows = []
    feature_count = len(selected_feature_order)

    for position, feature_name in enumerate(selected_feature_order, start=1):
        print(f"Standalone {position}/{feature_count}: {feature_name}")

        metrics = fit_and_evaluate_feature_subset([feature_name])
        feature_type = (
            "numeric"
            if feature_name in selected_numeric_features
            else "categorical"
        )

        standalone_rows.append({
            "feature": feature_name,
            "feature_type": feature_type,
            "standalone_MAE": metrics["MAE"],
            "standalone_RMSE": metrics["RMSE"],
            "standalone_R2": metrics["R2"],
            "standalone_MAE_gain": baseline_mae - metrics["MAE"],
        })

    standalone_feature_results = (
        pd.DataFrame(standalone_rows)
        .sort_values(
            ["standalone_MAE_gain", "feature"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )

    result_features = standalone_feature_results["feature"].tolist()
    if len(result_features) != feature_count:
        raise AssertionError("Standalone results must contain every selected feature once.")
    if set(result_features) != set(selected_feature_order):
        raise AssertionError("Standalone results do not match the selected features.")

    ranked_features = result_features

##### Standalone results

Features are ordered from the largest to the smallest standalone MAE gain.

In [ ]:
standalone_feature_results.round(4) if RUN_FEATURE_IMPACT_ANALYSIS else None

##### Plot standalone MAE gain

Green bars improve on the constant mean baseline; orange bars perform worse than it.

In [ ]:
if RUN_FEATURE_IMPACT_ANALYSIS:
    standalone_plot_data = standalone_feature_results.sort_values(
        "standalone_MAE_gain",
        ascending=True,
    ).reset_index(drop=True)
    standalone_colors = [
        COLOR_PRIMARY if gain >= 0 else COLOR_CORAL
        for gain in standalone_plot_data["standalone_MAE_gain"]
    ]
    y_positions = np.arange(len(standalone_plot_data))

    fig, ax = plt.subplots(
        figsize=(10, max(6, 0.4 * len(standalone_plot_data)))
    )
    ax.hlines(
        y_positions,
        0,
        standalone_plot_data["standalone_MAE_gain"],
        color=COLOR_LIGHT,
        linewidth=4,
    )
    ax.scatter(
        standalone_plot_data["standalone_MAE_gain"],
        y_positions,
        s=90,
        color=standalone_colors,
        edgecolor="white",
        linewidth=1,
        zorder=3,
    )
    ax.axvline(0, color=COLOR_NEUTRAL, linewidth=1, linestyle="--")
    for y_position, gain in zip(
        y_positions,
        standalone_plot_data["standalone_MAE_gain"],
    ):
        horizontal_alignment = "left"
        label_offset = 0.002
        ax.text(
            gain + label_offset,
            y_position,
            f"{gain:+.3f}",
            va="center",
            ha=horizontal_alignment,
            fontsize=8.5,
        )
    ax.set(
        xlabel="Standalone MAE gain versus constant mean baseline",
        ylabel="Feature",
        title=f"Standalone predictive value of each feature — {selected_model_name}",
        yticks=y_positions,
        yticklabels=standalone_plot_data["feature"],
    )
    ax.grid(axis="y", visible=False)

    plt.tight_layout()
    plt.show()

#### Add features cumulatively

Start with the strongest standalone feature, then add one feature at a time. `incremental_MAE_gain` measures the change caused by the latest addition, while `cumulative_MAE_gain` compares the current subset with the constant mean baseline.

In [ ]:
if RUN_FEATURE_IMPACT_ANALYSIS:
    cumulative_rows = []
    cumulative_features = []
    previous_mae = baseline_mae
    feature_count = len(ranked_features)

    for step, feature_name in enumerate(ranked_features, start=1):
        cumulative_features.append(feature_name)
        print(f"Cumulative {step}/{feature_count}: adding {feature_name}")

        metrics = fit_and_evaluate_feature_subset(cumulative_features)
        current_mae = metrics["MAE"]

        cumulative_rows.append({
            "step": step,
            "feature_added": feature_name,
            #"features_used": tuple(cumulative_features),
            "n_features": len(cumulative_features),
            "MAE": current_mae,
            "RMSE": metrics["RMSE"],
            "R2": metrics["R2"],
            "incremental_MAE_gain": previous_mae - current_mae,
            "cumulative_MAE_gain": baseline_mae - current_mae,
        })

        previous_mae = current_mae

    cumulative_feature_results = pd.DataFrame(cumulative_rows)

##### Validate the cumulative sequence

The subset must grow by exactly one feature per step and finish with every selected feature.

In [ ]:
if RUN_FEATURE_IMPACT_ANALYSIS:
    expected_feature_counts = list(range(1, len(selected_feature_order) + 1))
    actual_feature_counts = cumulative_feature_results["n_features"].tolist()

    if actual_feature_counts != expected_feature_counts:
        raise AssertionError("Cumulative feature counts must increase by exactly one.")
    if cumulative_features != ranked_features:
        raise AssertionError("Features were not added in standalone ranking order.")
    if set(cumulative_features) != set(selected_feature_order):
        raise AssertionError("The final subset does not contain every selected feature.")

##### Cumulative results

In [ ]:
cumulative_feature_results.round(4) if RUN_FEATURE_IMPACT_ANALYSIS else None

#### Check the full-feature result

Refitting the complete selected-model configuration. Its MAE should match the last cumulative step within the stated numerical tolerance.

In [ ]:
if RUN_FEATURE_IMPACT_ANALYSIS:
    print("Consistency fit: refitting the complete selected-model configuration.")

    full_subset_metrics = fit_and_evaluate_feature_subset(selected_feature_order)
    final_cumulative_mae = float(cumulative_feature_results.iloc[-1]["MAE"])
    fresh_full_mae = float(full_subset_metrics["MAE"])
    absolute_mae_difference = abs(final_cumulative_mae - fresh_full_mae)
    consistency_check_passed = bool(np.isclose(
        final_cumulative_mae,
        fresh_full_mae,
        rtol=FEATURE_IMPACT_MAE_TOLERANCE,
        atol=FEATURE_IMPACT_MAE_TOLERANCE,
    ))

    feature_impact_consistency = pd.DataFrame([{
        "model": selected_model_name,
        "features_match_complete_configuration": (
            cumulative_features == ranked_features
            and set(cumulative_features) == set(selected_feature_order)
        ),
        "final_cumulative_MAE": final_cumulative_mae,
        "fresh_full_configuration_MAE": fresh_full_mae,
        "absolute_MAE_difference": absolute_mae_difference,
        "consistency_check_passed": consistency_check_passed,
    }])

    if not consistency_check_passed:
        raise AssertionError(
            "Final cumulative MAE does not match a fresh full-configuration fit."
        )

##### Consistency-check result

In [ ]:
feature_impact_consistency.round(10) if RUN_FEATURE_IMPACT_ANALYSIS else None

#### Prepare the MAE waterfall

Convert the cumulative MAE path into step-by-step changes for the final chart.

In [ ]:
if RUN_FEATURE_IMPACT_ANALYSIS:
    cumulative_mae_values = cumulative_feature_results["MAE"].tolist()
    mae_path = [baseline_mae] + cumulative_mae_values
    waterfall_changes = [
        current_mae - previous_mae
        for previous_mae, current_mae in zip(mae_path[:-1], mae_path[1:])
    ]

    waterfall_labels = (
        ["Constant mean baseline"]
        + ranked_features
        + [f"Full {selected_model_name}"]
    )

    feature_hover_text = []
    for feature_name, mae_change, current_mae in zip(
        ranked_features,
        waterfall_changes,
        cumulative_mae_values,
    ):
        feature_hover_text.append(
            f"<b>{feature_name}</b><br>MAE change: {mae_change:+.6f}"
            f"<br>Test MAE after step: {current_mae:.6f}"
        )

    waterfall_hover_text = (
        [f"<b>Constant mean baseline</b><br>Test MAE: {baseline_mae:.6f}"]
        + feature_hover_text
        + [
            f"<b>Full {selected_model_name}</b>"
            f"<br>Test MAE: {final_cumulative_mae:.6f}"
        ]
    )

#### Plot the MAE waterfall

A downward step improves MAE; an upward step makes MAE worse at that point in the ranked sequence.

In [ ]:
if RUN_FEATURE_IMPACT_ANALYSIS:
    waterfall_measures = (
        ["absolute"]
        + ["relative"] * len(ranked_features)
        + ["total"]
    )
    waterfall_values = [baseline_mae] + waterfall_changes + [0]
    waterfall_text = (
        [f"{baseline_mae:.4f}"]
        + [f"{change:+.4f}" for change in waterfall_changes]
        + [f"{final_cumulative_mae:.4f}"]
    )

    feature_impact_waterfall = go.Figure(go.Waterfall(
        name="Test MAE",
        orientation="h",
        measure=waterfall_measures,
        y=waterfall_labels,
        x=waterfall_values,
        text=waterfall_text,
        textposition="outside",
        hovertext=waterfall_hover_text,
        hoverinfo="text",
        decreasing={"marker": {"color": COLOR_PRIMARY}},
        increasing={"marker": {"color": COLOR_CORAL}},
        totals={"marker": {"color": COLOR_BLUE}},
        connector={"line": {"color": "#9CA3AF", "width": 1}},
    ))
    feature_impact_waterfall.update_layout(
        title={
            "text": (
                "How ranked features change holdout MAE"
                f"<br><sup>{selected_model_name} · green improves MAE, coral worsens it</sup>"
            ),
            "x": 0.02,
            "xanchor": "left",
        },
        xaxis_title="Holdout MAE and step change",
        yaxis_title="",
        showlegend=False,
        template="plotly_white",
        height=max(720, 34 * len(waterfall_labels) + 130),
        waterfallgap=0.30,
        margin={"b": 70, "l": 210, "r": 80, "t": 90},
        hoverlabel={"align": "left"},
    )
    feature_impact_waterfall.update_xaxes(
        gridcolor="#E5E7EB",
        zeroline=False,
    )
    feature_impact_waterfall.update_yaxes(
        autorange="reversed",
        showgrid=False,
    )
    feature_impact_waterfall.show()

#### Interpretation

- **Standalone MAE gain** measures how useful a feature is by itself compared with the constant mean baseline.
- **Incremental MAE gain** measures what a feature adds after all higher-ranked features are already included.
- A feature can perform well alone but add little later when related features already contain similar information.
- Results can change with `selected_model_name` because model families use information differently.
- The cumulative path depends on the standalone ranking order.

This analysis measures predictive usefulness, not causal impact.

### 3.11 Hold-out prediction diagnostics

The plots below use the untouched test set. The dashed diagonal represents perfect predictions; the error distribution should be centred near zero without a strong skew.


#### Compare actual and predicted ratings

Inspecting held-out predictions and their errors to identify systematic over- or under-prediction.

In [ ]:
# Diagnose the selected model using the same held-out rows used for final evaluation.
y_test = y_model.loc[MODEL_TEST_INDEX]
best_test_predictions = predict_from_raw_input(
    best_fitted_record,
    X_model_raw.loc[MODEL_TEST_INDEX],
    feature_engineering_reference,
)

evaluation_details = pd.DataFrame({
    "actual_rating": y_test,
    "predicted_rating": best_test_predictions,
})
evaluation_details["prediction_error"] = (
    evaluation_details["predicted_rating"] - evaluation_details["actual_rating"]
)
evaluation_details["absolute_error"] = evaluation_details["prediction_error"].abs()

rating_min = min(
    evaluation_details["actual_rating"].min(),
    evaluation_details["predicted_rating"].min(),
)
rating_max = max(
    evaluation_details["actual_rating"].max(),
    evaluation_details["predicted_rating"].max(),
)
axis_padding = max((rating_max - rating_min) * 0.05, 0.02)
axis_min = max(0, rating_min - axis_padding)
axis_max = min(5, rating_max + axis_padding)

evaluation_details["actual_rating_bin"] = pd.qcut(
    evaluation_details["actual_rating"],
    q=10,
    duplicates="drop",
)
calibration_summary = (
    evaluation_details.groupby("actual_rating_bin", observed=True)
    .agg(
        actual_rating=("actual_rating", "mean"),
        predicted_rating=("predicted_rating", "mean"),
        books=("actual_rating", "size"),
    )
    .reset_index(drop=True)
)

fig, axes = plt.subplots(1, 3, figsize=(17, 5.3))
axes[0].scatter(
    evaluation_details["actual_rating"],
    evaluation_details["predicted_rating"],
    s=22,
    alpha=0.28,
    linewidth=0,
    color=COLOR_PRIMARY,
)
axes[0].plot(
    [axis_min, axis_max],
    [axis_min, axis_max],
    "--",
    color=COLOR_CORAL,
    linewidth=1.8,
    label="Perfect prediction",
)
axes[0].set(
    xlim=(axis_min, axis_max),
    ylim=(axis_min, axis_max),
    xlabel="Actual average rating",
    ylabel="Predicted average rating",
    title="Actual versus predicted ratings",
)
axes[0].legend(frameon=False)
axes[0].grid(False)

sns.histplot(
    data=evaluation_details,
    x="prediction_error",
    bins=32,
    kde=True,
    color=COLOR_BLUE,
    alpha=0.80,
    ax=axes[1],
)
axes[1].axvline(0, linestyle="--", color=COLOR_CORAL, linewidth=1.8, label="No error")
axes[1].axvline(
    evaluation_details["prediction_error"].median(),
    linestyle=":",
    color=COLOR_PRIMARY,
    linewidth=2,
    label=f"Median: {evaluation_details['prediction_error'].median():+.3f}",
)
axes[1].set(
    xlabel="Prediction error (predicted − actual)",
    ylabel="Books",
    title="Residual distribution",
)
axes[1].legend(frameon=False)
axes[1].grid(axis="x", visible=False)

axes[2].plot(
    calibration_summary["actual_rating"],
    calibration_summary["predicted_rating"],
    color=COLOR_PRIMARY,
    marker="o",
    linewidth=2.4,
    label="Mean prediction by actual-rating decile",
)
axes[2].plot(
    [axis_min, axis_max],
    [axis_min, axis_max],
    "--",
    color=COLOR_CORAL,
    linewidth=1.8,
    label="Perfect calibration",
)
axes[2].set(
    xlim=(axis_min, axis_max),
    ylim=(axis_min, axis_max),
    xlabel="Mean actual rating",
    ylabel="Mean predicted rating",
    title="Calibration across rating deciles",
)
axes[2].legend(frameon=False, fontsize=8.5)
axes[2].grid(axis="x", visible=False)

fig.suptitle(f"Holdout diagnostics — {selected_model_name}", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

evaluation_details.head(10)

#### Interpretation

The residual distribution is centered close to zero, but the calibration curve is flatter than the perfect-calibration line. The model therefore pulls predictions toward the dataset mean: lower-rated books tend to be overpredicted and higher-rated books tend to be underpredicted. This regression-to-the-mean pattern limits performance at the rating extremes even though most errors remain small.


------------
## 4. Extended Goodreads dataset
Source of the data : https://cseweb.ucsd.edu/~jmcauley/datasets/goodreads.html
With this extended dataset, containing the same fields as the original dataset, we can try to improve the model performance.

In this section, we prepare the larger SQLite source, and evaluate the already-selected model configuration on a separate train/test split. The extended variables remain separate from the original workflow and are also made available to the extended-model export.

### Extended-data imports and export controls

The cleaning workflow writes one reusable SQLite file on its first successful run. Later analysis validates and loads that file unless an explicit rebuild is requested.

In [ ]:
import os
import sqlite3
from contextlib import closing
from datetime import datetime, timezone
from pathlib import Path

EXTENDED_SOURCE_DB_PATH = Path("../data/external/goodreads_extended/goodreads_books_extended.db") # the dataset was pre-downloaded from the source
EXTENDED_CLEAN_DATASET_PATH = Path("../data/processed/goodreads_extended_cleaned.db")
EXTENDED_CLEAN_TEMP_PATH = Path("../data/processed/goodreads_extended_cleaned.tmp.db")
EXTENDED_CLEANING_VERSION = 1
EXTENDED_EDA_SAMPLE_SIZE = 100_000
REBUILD_EXTENDED_CLEAN_DATA = False

EXTENDED_REQUIRED_COLUMNS = {
    "bookID",
    "isbn",
    TARGET,
    *RAW_MODEL_INPUT_COLUMNS,
}

extended_export_controls = pd.DataFrame([
    {"setting": "source_database", "value": str(EXTENDED_SOURCE_DB_PATH)},
    {"setting": "cleaned_dataset_export", "value": str(EXTENDED_CLEAN_DATASET_PATH)},
    {"setting": "cleaning_version", "value": EXTENDED_CLEANING_VERSION},
    {"setting": "force_rebuild", "value": REBUILD_EXTENDED_CLEAN_DATA},
])

extended_export_controls

### Validate the saved cleaned dataset

The saved dataset is accepted only when its schema, row count, cleaning version, and available source fingerprint still match. A changed source or cleaning version requires an explicit rebuild.

In [ ]:
def extended_source_fingerprint(source_path):
    """Return the inexpensive file attributes used to detect a changed source."""
    source_path = Path(source_path)
    if not source_path.exists():
        return None

    source_stat = source_path.stat()
    return {
        "source_size_bytes": int(source_stat.st_size),
        "source_mtime_ns": int(source_stat.st_mtime_ns),
    }


def validate_extended_clean_export(export_path, source_path):
    """Validate the exported cleaned SQLite dataset and return its metadata."""
    export_path = Path(export_path)
    if not export_path.exists():
        raise FileNotFoundError(f"Cleaned dataset export not found: {export_path}")

    export_uri = f"file:{export_path.resolve().as_posix()}?mode=ro"
    with closing(sqlite3.connect(export_uri, uri=True)) as export_connection:
        integrity_result = export_connection.execute("PRAGMA quick_check").fetchone()[0]
        available_tables = {
            row[0]
            for row in export_connection.execute(
                "SELECT name FROM sqlite_master WHERE type = 'table'"
            )
        }
        required_tables = {"books_cleaned", "cleaning_metadata", "isbn_conflicts"}
        missing_tables = required_tables - available_tables
        if missing_tables:
            raise ValueError(f"Cleaned export is missing tables: {sorted(missing_tables)}")

        metadata = pd.read_sql_query("SELECT * FROM cleaning_metadata", export_connection)
        exported_row_count = export_connection.execute(
            "SELECT COUNT(*) FROM books_cleaned"
        ).fetchone()[0]

    if integrity_result != "ok":
        raise ValueError(f"Cleaned export integrity check failed: {integrity_result}")
    if len(metadata) != 1:
        raise ValueError("Cleaned export must contain exactly one metadata row.")

    metadata_row = metadata.iloc[0]
    if int(metadata_row["cleaning_version"]) != EXTENDED_CLEANING_VERSION:
        raise ValueError(
            "The cleaning rules changed. Set REBUILD_EXTENDED_CLEAN_DATA = True "
            "to rebuild the cleaned dataset."
        )
    if int(metadata_row["final_rows"]) != int(exported_row_count):
        raise ValueError("Cleaned export row count does not match its metadata.")

    current_fingerprint = extended_source_fingerprint(source_path)
    if current_fingerprint is not None:
        fingerprint_changed = any(
            int(metadata_row[field_name]) != current_fingerprint[field_name]
            for field_name in ["source_size_bytes", "source_mtime_ns"]
        )
        if fingerprint_changed:
            raise ValueError(
                "The extended source database changed. Set "
                "REBUILD_EXTENDED_CLEAN_DATA = True to recreate the export."
            )

    return metadata

### Apply deterministic extended-data cleaning

These rules correct source-specific invalid values without learning from the target or test set. Missing page counts and publication years remain missing so the existing train-fitted feature engineering can impute them later.

In [ ]:
def clean_extended_model_inputs(raw_data):
    """Apply deterministic cleaning while preserving rows and the raw model schema."""
    cleaned_data = raw_data.copy()

    for column_name in ["title", "authors", "publisher", "language_code"]:
        cleaned_data[column_name] = cleaned_data[column_name].astype("string").str.strip()

    cleaned_data["publisher"] = cleaned_data["publisher"].replace("", pd.NA)
    cleaned_data["language_code"] = cleaned_data["language_code"].replace("", pd.NA)
    cleaned_data["title"] = (
        cleaned_data["title"]
        .str.replace("�", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    cleaned_data[TARGET] = pd.to_numeric(cleaned_data[TARGET], errors="coerce")
    cleaned_data["ratings_count"] = (
        pd.to_numeric(cleaned_data["ratings_count"], errors="coerce")
        .fillna(0)
        .clip(lower=0)
    )
    cleaned_data["text_reviews_count"] = (
        pd.to_numeric(cleaned_data["text_reviews_count"], errors="coerce")
        .fillna(0)
        .clip(lower=0)
        .clip(upper=cleaned_data["ratings_count"])
    )

    page_counts = pd.to_numeric(cleaned_data["num_pages"], errors="coerce")
    invalid_page_count = (
        page_counts.isna()
        | page_counts.le(0)
        | page_counts.isin([9998, 9999])
        | page_counts.gt(10_000)
    )
    cleaned_data["num_pages"] = page_counts.mask(invalid_page_count)

    publication_date_text = cleaned_data["publication_date"].astype("string")
    publication_year = pd.to_numeric(
        publication_date_text.str.extract(
            r"(?<!\d)(\d{4})(?!\d)",
            expand=False,
        ),
        errors="coerce",
    )
    valid_publication_year = publication_year.between(1450, REFERENCE_YEAR)
    normalized_publication_year = publication_year.where(valid_publication_year).astype("Int64")
    cleaned_data["publication_date"] = pd.to_datetime(
        normalized_publication_year.astype("string") + "-01-01",
        errors="coerce",
    )

    return cleaned_data

### Resolve true duplicates within repeated ISBN groups

Repeated ISBN values are not automatically duplicate books because the source contains conflicting assignments. Only rows with the same normalized edition identity are collapsed; conflicting ISBN records are retained for audit.

In [ ]:
def normalize_extended_isbn(isbn_values):
    """Convert nullable integer ISBN values to zero-padded ISBN-10 strings."""
    numeric_isbn = pd.to_numeric(isbn_values, errors="coerce").astype("Int64")
    return numeric_isbn.astype("string").str.zfill(10)


def resolve_extended_isbn_duplicates(cleaned_data):
    """Remove repeated snapshots of one edition and retain ISBN conflicts."""
    resolved_data = cleaned_data.copy()
    resolved_data["isbn"] = normalize_extended_isbn(resolved_data["isbn"])

    resolved_data["_identity_title"] = resolved_data["title"].map(normalize_model_text)
    resolved_data["_identity_authors"] = resolved_data["authors"].map(normalize_model_text)
    resolved_data["_identity_publisher"] = resolved_data["publisher"].map(normalize_model_text)
    resolved_data["_identity_year"] = resolved_data["publication_date"].dt.year.astype("Int64")
    resolved_data["_identity_pages"] = resolved_data["num_pages"].astype("Float64")

    identity_columns = [
        "isbn",
        "_identity_title",
        "_identity_authors",
        "_identity_publisher",
        "_identity_year",
        "_identity_pages",
    ]
    repeated_isbn_mask = (
        resolved_data["isbn"].notna()
        & resolved_data.duplicated("isbn", keep=False)
    )
    repeated_isbn_groups = int(resolved_data.loc[repeated_isbn_mask, "isbn"].nunique())

    ranked_candidates = resolved_data.loc[repeated_isbn_mask].sort_values(
        ["ratings_count", "text_reviews_count", "bookID"],
        ascending=[False, False, True],
    )
    repeated_snapshot_mask = ranked_candidates.duplicated(identity_columns, keep="first")
    duplicate_book_ids = ranked_candidates.loc[repeated_snapshot_mask, "bookID"]
    resolved_data = resolved_data.loc[
        ~resolved_data["bookID"].isin(duplicate_book_ids)
    ].copy()

    remaining_identity_duplicates = (
        resolved_data["isbn"].notna()
        & resolved_data.duplicated(identity_columns, keep=False)
    )
    if remaining_identity_duplicates.any():
        raise AssertionError("True ISBN edition duplicates remain after resolution.")

    conflict_mask = (
        resolved_data["isbn"].notna()
        & resolved_data.duplicated("isbn", keep=False)
    )
    conflict_columns = [
        "isbn",
        "bookID",
        "title",
        "authors",
        "publisher",
        "publication_date",
        "num_pages",
        "average_rating",
        "ratings_count",
        "text_reviews_count",
    ]
    isbn_conflicts = (
        resolved_data.loc[conflict_mask, conflict_columns]
        .sort_values(["isbn", "bookID"])
        .reset_index(drop=True)
    )

    identity_helper_columns = [
        column_name
        for column_name in resolved_data.columns
        if column_name.startswith("_identity_")
    ]
    resolved_data = resolved_data.drop(columns=identity_helper_columns)

    duplicate_audit = {
        "repeated_isbn_groups": repeated_isbn_groups,
        "true_duplicate_rows_removed": int(len(duplicate_book_ids)),
        "isbn_conflict_groups": int(isbn_conflicts["isbn"].nunique()),
        "isbn_conflict_rows": int(len(isbn_conflicts)),
    }
    return resolved_data, isbn_conflicts, duplicate_audit

### Load and clean the raw source once

The raw database is scanned only while creating or deliberately recreating the cleaned export. Eligibility checks are reported before deterministic cleaning and ISBN resolution.

In [ ]:
def load_and_clean_extended_source(source_path):
    """Load eligible source rows, apply cleaning, and prepare export metadata."""
    source_path = Path(source_path)
    if not source_path.exists():
        raise FileNotFoundError(f"Extended source database not found: {source_path}")

    source_uri = f"file:{source_path.resolve().as_posix()}?mode=ro"
    with sqlite3.connect(source_uri, uri=True) as source_connection:
        integrity_result = source_connection.execute("PRAGMA quick_check").fetchone()[0]
        if integrity_result != "ok":
            raise ValueError(f"Source database integrity check failed: {integrity_result}")

        source_columns = {
            row[1]
            for row in source_connection.execute("PRAGMA table_info(books)")
        }
        missing_columns = EXTENDED_REQUIRED_COLUMNS - source_columns
        if missing_columns:
            raise ValueError(f"Extended source is missing columns: {sorted(missing_columns)}")

        source_profile_query = """
        SELECT
            COUNT(*) AS source_rows,
            SUM(ratings_count >= ?) AS eligible_rating_count_rows,
            SUM(ratings_count = 0 AND average_rating = 0) AS unrated_rows,
            SUM(language_code IS NULL OR TRIM(language_code) = '') AS source_language_missing_rows,
            SUM(publisher IS NULL OR TRIM(publisher) = '') AS source_publisher_missing_rows,
            SUM(isbn IS NULL) AS source_isbn_missing_rows
        FROM books
        """
        source_profile = pd.read_sql_query(
            source_profile_query,
            source_connection,
            params=[MIN_RATINGS_COUNT_FOR_MODELLING],
        ).iloc[0]

        eligible_query = """
        SELECT
            bookID,
            isbn,
            average_rating,
            title,
            authors,
            language_code,
            num_pages,
            ratings_count,
            text_reviews_count,
            publication_date,
            publisher
        FROM books
        WHERE ratings_count >= ?
        """
        eligible_data = pd.read_sql_query(
            eligible_query,
            source_connection,
            params=[MIN_RATINGS_COUNT_FOR_MODELLING],
        )

    numeric_target = pd.to_numeric(eligible_data[TARGET], errors="coerce")
    title_text = eligible_data["title"].astype("string").str.strip()
    author_text = eligible_data["authors"].astype("string").str.strip()

    invalid_target_mask = ~numeric_target.between(0, 5)
    blank_title_mask = title_text.isna() | title_text.eq("")
    blank_author_mask = author_text.isna() | author_text.eq("")
    invalid_author_mask = author_text.str.lower().eq("not a book").fillna(False)
    removal_mask = (
        invalid_target_mask
        | blank_title_mask
        | blank_author_mask
        | invalid_author_mask
    )

    eligible_cleaning_input = eligible_data.loc[~removal_mask].copy()
    raw_page_counts = pd.to_numeric(
        eligible_cleaning_input["num_pages"],
        errors="coerce",
    )
    page_values_set_missing = (
        raw_page_counts.isna()
        | raw_page_counts.le(0)
        | raw_page_counts.isin([9998, 9999])
        | raw_page_counts.gt(10_000)
    )
    replacement_title_rows = int(
        eligible_cleaning_input["title"].astype("string").str.contains("�", na=False).sum()
    )
    possible_mojibake_title_rows = int(
        eligible_cleaning_input["title"]
        .astype("string")
        .str.contains("Ã|Â", regex=True, na=False)
        .sum()
    )

    cleaned_data = clean_extended_model_inputs(eligible_cleaning_input)
    invalid_publication_year_rows = int(cleaned_data["publication_date"].isna().sum())
    resolved_data, isbn_conflicts, duplicate_audit = resolve_extended_isbn_duplicates(
        cleaned_data
    )

    source_fingerprint = extended_source_fingerprint(source_path)
    cleaning_metadata = {
        "cleaning_version": EXTENDED_CLEANING_VERSION,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_path": str(EXTENDED_SOURCE_DB_PATH),
        "source_size_bytes": source_fingerprint["source_size_bytes"],
        "source_mtime_ns": source_fingerprint["source_mtime_ns"],
        "source_rows": int(source_profile["source_rows"]),
        "eligible_rating_count_rows": int(source_profile["eligible_rating_count_rows"]),
        "unrated_rows": int(source_profile["unrated_rows"]),
        "eligible_rows_loaded": int(len(eligible_data)),
        "invalid_target_rows_removed": int(invalid_target_mask.sum()),
        "blank_title_rows_removed": int(blank_title_mask.sum()),
        "blank_author_rows_removed": int(blank_author_mask.sum()),
        "invalid_author_rows_removed": int(invalid_author_mask.sum()),
        "page_values_set_missing": int(page_values_set_missing.sum()),
        "invalid_publication_year_rows": invalid_publication_year_rows,
        "replacement_title_rows": replacement_title_rows,
        "possible_mojibake_title_rows": possible_mojibake_title_rows,
        "language_missing_rows": int(resolved_data["language_code"].isna().sum()),
        "publisher_missing_rows": int(resolved_data["publisher"].isna().sum()),
        "isbn_missing_rows": int(resolved_data["isbn"].isna().sum()),
        "minimum_ratings_count": MIN_RATINGS_COUNT_FOR_MODELLING,
        "reference_year": REFERENCE_YEAR,
        **duplicate_audit,
        "final_rows": int(len(resolved_data)),
    }

    export_columns = ["bookID", "isbn", TARGET, *RAW_MODEL_INPUT_COLUMNS]
    resolved_data = resolved_data[export_columns].sort_values("bookID").reset_index(drop=True)
    return resolved_data, isbn_conflicts, cleaning_metadata

### Export the cleaned dataset safely

Writing first to a temporary database protects the existing cleaned file if the operation is interrupted. The final file is replaced only after table, index, and integrity checks succeed.

In [ ]:
def export_extended_clean_dataset(cleaned_data, isbn_conflicts, cleaning_metadata):
    """Write cleaned tables to a temporary database and replace the export."""
    EXTENDED_CLEAN_DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
    if EXTENDED_CLEAN_TEMP_PATH.exists():
        EXTENDED_CLEAN_TEMP_PATH.unlink()

    try:
        with closing(sqlite3.connect(EXTENDED_CLEAN_TEMP_PATH)) as export_connection:
            cleaned_data.to_sql(
                "books_cleaned",
                export_connection,
                if_exists="replace",
                index=False,
            )
            pd.DataFrame([cleaning_metadata]).to_sql(
                "cleaning_metadata",
                export_connection,
                if_exists="replace",
                index=False,
            )
            isbn_conflicts.to_sql(
                "isbn_conflicts",
                export_connection,
                if_exists="replace",
                index=False,
            )
            export_connection.execute(
                "CREATE UNIQUE INDEX idx_books_cleaned_bookID "
                "ON books_cleaned(bookID)"
            )
            export_connection.execute(
                "CREATE INDEX idx_books_cleaned_isbn ON books_cleaned(isbn)"
            )
            export_connection.execute("ANALYZE")
            integrity_result = export_connection.execute("PRAGMA quick_check").fetchone()[0]
            if integrity_result != "ok":
                raise ValueError(f"Temporary export integrity check failed: {integrity_result}")

        os.replace(EXTENDED_CLEAN_TEMP_PATH, EXTENDED_CLEAN_DATASET_PATH)
    except Exception:
        if EXTENDED_CLEAN_TEMP_PATH.exists():
            EXTENDED_CLEAN_TEMP_PATH.unlink()
        raise

### Run the guarded cleaning export

Running the cleaning workflow only when the saved dataset is absent or an explicit rebuild is requested. Otherwise, validating the existing export without rereading the raw books table.

In [ ]:
extended_clean_export_was_created = False

if EXTENDED_CLEAN_DATASET_PATH.exists() and not REBUILD_EXTENDED_CLEAN_DATA:
    extended_cleaning_metadata = validate_extended_clean_export(
        EXTENDED_CLEAN_DATASET_PATH,
        EXTENDED_SOURCE_DB_PATH,
    )
    extended_export_status = "validated existing cleaned dataset export"
else:
    (
        extended_build_data,
        extended_build_isbn_conflicts,
        extended_build_metadata,
    ) = load_and_clean_extended_source(EXTENDED_SOURCE_DB_PATH)
    export_extended_clean_dataset(
        extended_build_data,
        extended_build_isbn_conflicts,
        extended_build_metadata,
    )
    extended_cleaning_metadata = validate_extended_clean_export(
        EXTENDED_CLEAN_DATASET_PATH,
        EXTENDED_SOURCE_DB_PATH,
    )
    extended_clean_export_was_created = True
    extended_export_status = "created cleaned dataset export from raw source"
    del extended_build_data, extended_build_isbn_conflicts, extended_build_metadata

extended_export_status_summary = pd.DataFrame([{
    "status": extended_export_status,
    "cleaning_version": int(extended_cleaning_metadata.loc[0, "cleaning_version"]),
    "exported_rows": int(extended_cleaning_metadata.loc[0, "final_rows"]),
    "export_size_mb": EXTENDED_CLEAN_DATASET_PATH.stat().st_size / (1024 ** 2),
}])

extended_export_status_summary.round({"export_size_mb": 1})

#### Interpretation

The status table confirms whether this run created the cleaned export or only validated it. Loading and analysis are deliberately separated from the one-time cleaning pipeline.

### Load the saved cleaned dataset

All following EDA and modelling use the exported `books_cleaned` table directly. This cell does not access the raw extended-data table or repeat any cleaning.

In [ ]:
clean_export_uri = f"file:{EXTENDED_CLEAN_DATASET_PATH.resolve().as_posix()}?mode=ro"
with closing(sqlite3.connect(clean_export_uri, uri=True)) as clean_export_connection:
    extended_model_source_df = pd.read_sql_query(
        "SELECT * FROM books_cleaned ORDER BY bookID",
        clean_export_connection,
        parse_dates=["publication_date"],
    )

expected_extended_rows = int(extended_cleaning_metadata.loc[0, "final_rows"])
if len(extended_model_source_df) != expected_extended_rows:
    raise AssertionError("Loaded cleaned rows do not match the export metadata.")

extended_load_summary = pd.DataFrame([{
    "dataset": EXTENDED_CLEAN_DATASET_PATH.name,
    "table": "books_cleaned",
    "rows_loaded": len(extended_model_source_df),
    "columns_loaded": extended_model_source_df.shape[1],
}])
extended_load_summary

#### Interpretation

The row-count check links the loaded modelling data to the audit stored during export, making the saved dataset the explicit boundary between cleaning and analysis.

### Review the stored cleaning audit

Keeping the audit inside the cleaned database makes the one-time decisions visible on later runs without rescanning the raw source.

In [ ]:
extended_audit_fields = [
    "source_rows",
    "eligible_rows_loaded",
    "invalid_target_rows_removed",
    "blank_title_rows_removed",
    "blank_author_rows_removed",
    "invalid_author_rows_removed",
    "page_values_set_missing",
    "invalid_publication_year_rows",
    "replacement_title_rows",
    "possible_mojibake_title_rows",
    "repeated_isbn_groups",
    "true_duplicate_rows_removed",
    "isbn_conflict_groups",
    "isbn_conflict_rows",
    "final_rows",
]
extended_cleaning_audit = (
    extended_cleaning_metadata[extended_audit_fields]
    .T
    .reset_index()
    .rename(columns={"index": "metric", 0: "value"})
)

extended_cleaning_audit

#### Interpretation

The audit separates removed records from corrected or missing values. ISBN conflicts remain available in their own table because sharing a corrupted identifier does not make two different books duplicates.

### Compare the original and extended modelling populations

A compact comparison highlights the scale and engagement-distribution changes that may affect the selected model.

In [ ]:
def summarize_modelling_population(data, dataset_name):
    language_missing = (
        data["language_code"].isna()
        | data["language_code"].astype("string").str.strip().eq("")
    )
    publisher_missing = (
        data["publisher"].isna()
        | data["publisher"].astype("string").str.strip().eq("")
    )
    return {
        "dataset": dataset_name,
        "rows": len(data),
        "mean_rating": data[TARGET].mean(),
        "median_rating": data[TARGET].median(),
        "median_ratings_count": data["ratings_count"].median(),
        "median_text_reviews_count": data["text_reviews_count"].median(),
        "median_num_pages": data["num_pages"].median(),
        "num_pages_missing_share": data["num_pages"].isna().mean(),
        "language_missing_share": language_missing.mean(),
        "publisher_missing_share": publisher_missing.mean(),
    }


extended_dataset_comparison = pd.DataFrame([
    summarize_modelling_population(model_source_df, "Original modelling data"),
    summarize_modelling_population(extended_model_source_df, "Extended cleaned data"),
])

extended_dataset_comparison.round({
    "mean_rating": 3,
    "median_rating": 3,
    "median_ratings_count": 1,
    "median_text_reviews_count": 1,
    "median_num_pages": 1,
    "num_pages_missing_share": 3,
    "language_missing_share": 3,
    "publisher_missing_share": 3,
})

#### Interpretation

Differences in rating support, written-review activity, and metadata availability indicate that the extended result should be treated as a separate evaluation rather than a direct continuation of the original score.

### Compare cleaned distributions visually

Density-normalized plots compare distribution shapes rather than raw dataset sizes. Each dataset is sampled reproducibly, with at most 100,000 rows.

In [ ]:
original_eda_sample = model_source_df.sample(
    n=min(EXTENDED_EDA_SAMPLE_SIZE, len(model_source_df)),
    random_state=RANDOM_STATE,
).copy()
original_eda_sample["dataset"] = "Original"

extended_eda_sample = extended_model_source_df.sample(
    n=min(EXTENDED_EDA_SAMPLE_SIZE, len(extended_model_source_df)),
    random_state=RANDOM_STATE,
).copy()
extended_eda_sample["dataset"] = "Extended cleaned"

comparison_plot_data = pd.concat(
    [original_eda_sample, extended_eda_sample],
    ignore_index=True,
)
comparison_plot_data["log_ratings_count"] = np.log1p(
    comparison_plot_data["ratings_count"]
)
comparison_plot_data["log_text_reviews_count"] = np.log1p(
    comparison_plot_data["text_reviews_count"]
)
page_plot_limit = comparison_plot_data["num_pages"].quantile(0.995)
page_plot_data = comparison_plot_data.loc[
    comparison_plot_data["num_pages"].le(page_plot_limit)
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
plot_specs = [
    (comparison_plot_data, TARGET, "Average-rating distribution", "Average rating"),
    (comparison_plot_data, "log_ratings_count", "Rating support", "log1p(ratings_count)"),
    (comparison_plot_data, "log_text_reviews_count", "Written-review activity", "log1p(text_reviews_count)"),
    (page_plot_data, "num_pages", "Page counts below the combined 99.5th percentile", "Pages"),
]
for plot_index, (axis, (plot_data, column_name, title, x_label)) in enumerate(
    zip(axes.flat, plot_specs)
):
    sns.ecdfplot(
        data=plot_data,
        x=column_name,
        hue="dataset",
        palette=DATASET_PALETTE,
        linewidth=2.2,
        ax=axis,
    )
    axis.set_title(title)
    axis.set_xlabel(x_label)
    axis.set_ylabel("Cumulative share of books")
    axis.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
    axis.grid(axis="x", visible=False)
    if plot_index > 0 and axis.get_legend() is not None:
        axis.get_legend().remove()
    elif axis.get_legend() is not None:
        axis.get_legend().set_title("")
        axis.get_legend().set_frame_on(False)

fig.suptitle("How the extended cleaned population shifts the original distributions", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

#### Interpretation

The overlays show whether the cleaned extended population changes the target, engagement, or book-length distributions seen in the original modelling data. They are generated from the saved export and do not rerun cleaning.

### Create the extended train/test split

Applying the existing stratified split before fitting any learned transformation. The holdout is the test partition kept separate from model fitting.

In [ ]:
extended_model_source_indexed = extended_model_source_df.set_index("bookID", drop=True)
extended_X_model_raw = extended_model_source_indexed[RAW_MODEL_INPUT_COLUMNS].copy()
extended_y_model = pd.to_numeric(
    extended_model_source_indexed[TARGET],
    errors="coerce",
)

if extended_y_model.isna().any():
    raise AssertionError("The cleaned extended target contains missing values.")

(
    EXTENDED_MODEL_TRAIN_INDEX,
    EXTENDED_MODEL_TEST_INDEX,
    extended_model_rating_bands,
) = create_stratified_rating_split(extended_y_model)

extended_split_summary = pd.DataFrame([
    {
        "split": "train",
        "books": len(EXTENDED_MODEL_TRAIN_INDEX),
        "mean_rating": extended_y_model.loc[EXTENDED_MODEL_TRAIN_INDEX].mean(),
        "median_rating": extended_y_model.loc[EXTENDED_MODEL_TRAIN_INDEX].median(),
        "rating_std": extended_y_model.loc[EXTENDED_MODEL_TRAIN_INDEX].std(),
    },
    {
        "split": "test",
        "books": len(EXTENDED_MODEL_TEST_INDEX),
        "mean_rating": extended_y_model.loc[EXTENDED_MODEL_TEST_INDEX].mean(),
        "median_rating": extended_y_model.loc[EXTENDED_MODEL_TEST_INDEX].median(),
        "rating_std": extended_y_model.loc[EXTENDED_MODEL_TEST_INDEX].std(),
    },
])
extended_split_summary["share_of_data"] = (
    extended_split_summary["books"] / len(extended_y_model)
)

extended_split_summary.round(3)

#### Interpretation

The split summary checks that the larger holdout keeps the intended size and a target distribution comparable with the training partition.

### Check extended rating-band balance

Comparing band shares confirms that the stratification rule used for the original experiment remains effective at the larger scale.

In [ ]:
extended_split_band_data = pd.concat([
    pd.DataFrame({
        "split": "train",
        "rating_band": extended_model_rating_bands.loc[
            EXTENDED_MODEL_TRAIN_INDEX
        ].astype(str),
    }),
    pd.DataFrame({
        "split": "test",
        "rating_band": extended_model_rating_bands.loc[
            EXTENDED_MODEL_TEST_INDEX
        ].astype(str),
    }),
])
extended_rating_band_split_share = pd.crosstab(
    extended_split_band_data["rating_band"],
    extended_split_band_data["split"],
    normalize="columns",
).reindex(columns=["train", "test"])

(extended_rating_band_split_share * 100).round(2)

#### Interpretation

Similar train and test percentages across the rating bands support using the same holdout metrics for the standalone extended-data evaluation.

### Fit extended feature engineering on training rows

Publication and page imputations, language groups, category frequencies, and target-derived statistics are learned from the extended training partition only.

In [ ]:
extended_feature_engineering_reference = fit_feature_engineering_reference(
    extended_X_model_raw.loc[EXTENDED_MODEL_TRAIN_INDEX],
    extended_y_model.loc[EXTENDED_MODEL_TRAIN_INDEX],
)
extended_X_train_engineered = apply_feature_engineering(
    extended_X_model_raw.loc[EXTENDED_MODEL_TRAIN_INDEX],
    extended_feature_engineering_reference,
)
extended_X_test_engineered = apply_feature_engineering(
    extended_X_model_raw.loc[EXTENDED_MODEL_TEST_INDEX],
    extended_feature_engineering_reference,
)

extended_selected_feature_names = sorted({
    feature_name
    for feature_group in [
        MODEL_SPECS[selected_model_name]["numeric_features"],
        MODEL_SPECS[selected_model_name]["categorical_features"],
    ]
    for feature_name in feature_group
})
extended_missing_selected_features = [
    feature_name
    for feature_name in extended_selected_feature_names
    if feature_name not in extended_X_train_engineered.columns
]
if extended_missing_selected_features:
    raise AssertionError(
        f"Extended feature engineering is missing: {extended_missing_selected_features}"
    )

extended_numeric_engineered = extended_X_train_engineered.select_dtypes(include=[np.number])
if np.isinf(extended_numeric_engineered.to_numpy(dtype=float)).any():
    raise AssertionError("Extended engineered training data contains infinite values.")

extended_feature_engineering_summary = pd.DataFrame([{
    "train_rows": len(extended_X_train_engineered),
    "test_rows": len(extended_X_test_engineered),
    "engineered_columns": extended_X_train_engineered.shape[1],
    "selected_features": len(extended_selected_feature_names),
    "missing_selected_features": len(extended_missing_selected_features),
}])

extended_feature_engineering_summary

#### Interpretation

The validation confirms that the existing modelling pipeline produces every feature required by the selected model without introducing infinite numeric values.

### Check overlap with the original holdout

Measuring shared ISBNs before interpreting transfer performance. An ISBN-disjoint subset contains only original holdout books whose ISBN does not appear in extended training.

In [ ]:
original_holdout_isbn = normalize_extended_isbn(
    model_source_df.loc[MODEL_TEST_INDEX, "isbn"]
)
extended_training_isbn = set(
    extended_model_source_indexed
    .loc[EXTENDED_MODEL_TRAIN_INDEX, "isbn"]
    .dropna()
    .astype(str)
)
original_holdout_isbn_overlap = original_holdout_isbn.isin(
    extended_training_isbn
)
original_holdout_verified_disjoint_index = original_holdout_isbn.index[
    original_holdout_isbn.notna() & ~original_holdout_isbn_overlap
]
if len(original_holdout_verified_disjoint_index) == 0:
    raise ValueError("No ISBN-disjoint original holdout rows remain.")

original_holdout_overlap_summary = pd.DataFrame([{
    "original_holdout_rows": len(MODEL_TEST_INDEX),
    "rows_with_normalized_isbn": int(original_holdout_isbn.notna().sum()),
    "isbn_overlap_rows": int(original_holdout_isbn_overlap.sum()),
    "isbn_overlap_share_all_rows": original_holdout_isbn_overlap.mean(),
    "verified_isbn_disjoint_rows": len(original_holdout_verified_disjoint_index),
}])

original_holdout_overlap_summary.round({
    "isbn_overlap_share_all_rows": 3,
})

#### Interpretation

An all-row original-holdout score is useful for comparison but is not a fully independent transfer test when ISBNs overlap with extended training. The verified disjoint subset provides a stricter secondary view.

### Evaluate the selected XGBoost configuration across datasets

Fitting the original winning configuration on the extended training split and evaluating it without another comparison or tuning pass.

In [ ]:
if selected_model_name != "XGBoost":
    raise ValueError(
        f"The saved selected model is {selected_model_name!r}, not the expected XGBoost."
    )

extended_selected_configs = [
    config
    for config in active_model_configs
    if config["model"] == selected_model_name
]
if len(extended_selected_configs) != 1:
    raise ValueError(
        f"Expected one active configuration for {selected_model_name!r}; "
        f"found {len(extended_selected_configs)}."
    )

extended_selected_config = extended_selected_configs[0]
expected_extended_xgboost_parameters = {
    "n_estimators": 400,
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 5.0,
    "tree_method": "hist",
    "random_state": RANDOM_STATE,
}
actual_extended_xgboost_parameters = extended_selected_config["estimator"].get_params()
parameter_mismatches = {
    parameter_name: {
        "expected": expected_value,
        "actual": actual_extended_xgboost_parameters.get(parameter_name),
    }
    for parameter_name, expected_value in expected_extended_xgboost_parameters.items()
    if actual_extended_xgboost_parameters.get(parameter_name) != expected_value
}
if parameter_mismatches:
    raise AssertionError(f"Selected XGBoost settings changed: {parameter_mismatches}")

extended_fitted_record = fit_model_configuration(
    extended_selected_config,
    extended_X_train_engineered,
    extended_y_model.loc[EXTENDED_MODEL_TRAIN_INDEX],
)
extended_test_predictions = predict_from_raw_input(
    extended_fitted_record,
    extended_X_model_raw.loc[EXTENDED_MODEL_TEST_INDEX],
    extended_feature_engineering_reference,
)
extended_metrics = regression_metrics(
    extended_y_model.loc[EXTENDED_MODEL_TEST_INDEX],
    extended_test_predictions,
)

extended_on_original_predictions = predict_from_raw_input(
    extended_fitted_record,
    X_model_raw.loc[MODEL_TEST_INDEX],
    extended_feature_engineering_reference,
)
extended_on_original_metrics = regression_metrics(
    y_model.loc[MODEL_TEST_INDEX],
    extended_on_original_predictions,
)
extended_on_original_prediction_series = pd.Series(
    extended_on_original_predictions,
    index=MODEL_TEST_INDEX,
)
extended_on_original_disjoint_metrics = regression_metrics(
    y_model.loc[original_holdout_verified_disjoint_index],
    extended_on_original_prediction_series.loc[
        original_holdout_verified_disjoint_index
    ],
)
original_selected_metrics = regression_metrics(
    y_model.loc[MODEL_TEST_INDEX],
    best_test_predictions,
)

extended_model_result = pd.DataFrame([
    {
        "trained_on": "Original cleaned data",
        "evaluated_on": "Original holdout",
        "model": selected_model_name,
        "train_rows": len(MODEL_TRAIN_INDEX),
        "test_rows": len(MODEL_TEST_INDEX),
        **original_selected_metrics,
    },
    {
        "trained_on": "Extended cleaned data",
        "evaluated_on": "Extended holdout",
        "model": selected_model_name,
        "train_rows": len(EXTENDED_MODEL_TRAIN_INDEX),
        "test_rows": len(EXTENDED_MODEL_TEST_INDEX),
        **extended_metrics,
    },
    {
        "trained_on": "Extended cleaned data",
        "evaluated_on": "Original holdout",
        "model": selected_model_name,
        "train_rows": len(EXTENDED_MODEL_TRAIN_INDEX),
        "test_rows": len(MODEL_TEST_INDEX),
        **extended_on_original_metrics,
    },
    {
        "trained_on": "Extended cleaned data",
        "evaluated_on": "Original holdout (ISBN-disjoint)",
        "model": selected_model_name,
        "train_rows": len(EXTENDED_MODEL_TRAIN_INDEX),
        "test_rows": len(original_holdout_verified_disjoint_index),
        **extended_on_original_disjoint_metrics,
    },
])

extended_model_result.round({"MAE": 3, "RMSE": 3, "R2": 3})

#### Visualize original and extended evaluation scenarios

The left panel compares error in rating points; the right panel shows explained variance. The ISBN-disjoint scenario is the strictest transfer check.

In [ ]:
extended_model_plot = extended_model_result.copy()
extended_model_plot["scenario"] = [
    "Original → original holdout",
    "Extended → extended holdout",
    "Extended → original holdout",
    "Extended → ISBN-disjoint holdout",
]
extended_model_plot = extended_model_plot.iloc[::-1].reset_index(drop=True)
y_positions = np.arange(len(extended_model_plot))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.3), sharey=True)
axes[0].hlines(
    y_positions,
    extended_model_plot["MAE"],
    extended_model_plot["RMSE"],
    color=COLOR_LIGHT,
    linewidth=5,
)
axes[0].scatter(
    extended_model_plot["MAE"],
    y_positions,
    s=115,
    color=COLOR_PRIMARY,
    edgecolor="white",
    linewidth=1,
    label="MAE",
    zorder=3,
)
axes[0].scatter(
    extended_model_plot["RMSE"],
    y_positions,
    s=115,
    color=COLOR_GOLD,
    edgecolor="white",
    linewidth=1,
    marker="D",
    label="RMSE",
    zorder=3,
)
for y_position, row in extended_model_plot.iterrows():
    axes[0].text(
        row["MAE"], y_position + 0.14, f"{row['MAE']:.3f}",
        ha="center", va="bottom", fontsize=8.5,
    )
    axes[0].text(
        row["RMSE"], y_position - 0.14, f"{row['RMSE']:.3f}",
        ha="center", va="top", fontsize=8.5,
    )
axes[0].set(
    title="Error metrics (lower is better)",
    xlabel="Rating points",
    ylabel="Training → evaluation scenario",
    yticks=y_positions,
    yticklabels=extended_model_plot["scenario"],
)
axes[0].legend(frameon=False, ncol=2, loc="upper right", bbox_to_anchor=(1, 1.12))
axes[0].grid(axis="y", visible=False)

axes[1].hlines(
    y_positions,
    0,
    extended_model_plot["R2"],
    color=COLOR_LIGHT,
    linewidth=5,
)
axes[1].scatter(
    extended_model_plot["R2"],
    y_positions,
    s=120,
    color=COLOR_BLUE,
    edgecolor="white",
    linewidth=1,
    zorder=3,
)
for y_position, r2_value in zip(y_positions, extended_model_plot["R2"]):
    axes[1].text(r2_value + 0.015, y_position, f"{r2_value:.3f}", va="center", fontsize=8.5)
axes[1].set(
    title="Explained variance (higher is better)",
    xlabel="R²",
    yticks=y_positions,
    yticklabels=extended_model_plot["scenario"],
)
axes[1].grid(axis="y", visible=False)

fig.suptitle("Extended-data performance across evaluation scenarios", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

#### Interpretation

The first two rows provide within-dataset references. The complete original holdout includes ISBN overlap with extended training, while the ISBN-disjoint row is the stricter transfer estimate. Section 5 can optionally refit this configuration on all eligible extended rows and export it with an `_extended` suffix.

------------
## 5. Model export

Refitting the selected model on all eligible rows from the chosen dataset and saving one self-contained Python predictor with `cloudpickle`, a library that stores fitted Python objects for later reuse.

### 5.1 Export controls

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import platform

import catboost as catboost_package
import cloudpickle
import sklearn as sklearn_package
import xgboost as xgboost_package

EXPORT_MODEL = True
OVERWRITE_MODEL = True
EXPORT_TRAINING_DATASET = "extended"  # Allowed values: "original" or "extended".
MODEL_TO_EXPORT = best_final_model["model"]

if EXPORT_TRAINING_DATASET == "original":
    export_X_raw = X_model_raw
    export_y = y_model
    export_artifact_suffix = ""
elif EXPORT_TRAINING_DATASET == "extended":
    required_extended_variables = ["extended_X_model_raw", "extended_y_model"]
    missing_extended_variables = [
        variable_name
        for variable_name in required_extended_variables
        if variable_name not in globals()
    ]
    if missing_extended_variables:
        raise RuntimeError(
            "Run Section 4 before selecting the extended dataset for export. "
            f"Missing variables: {missing_extended_variables}"
        )
    export_X_raw = extended_X_model_raw
    export_y = extended_y_model
    export_artifact_suffix = "_extended"
else:
    raise ValueError(
        "EXPORT_TRAINING_DATASET must be either 'original' or 'extended'."
    )

if not export_X_raw.index.equals(export_y.index):
    raise AssertionError("Export features and target do not have matching indexes.")

MODEL_EXPORT_PATH = Path(
    f"../models/goodreads_rating_predictor_{MODEL_TO_EXPORT}"
    f"{export_artifact_suffix}.pkl"
)

export_controls = pd.DataFrame([
    {"setting": "export_enabled", "value": EXPORT_MODEL},
    {"setting": "overwrite_enabled", "value": OVERWRITE_MODEL},
    {"setting": "training_dataset", "value": EXPORT_TRAINING_DATASET},
    {"setting": "training_rows", "value": len(export_X_raw)},
    {"setting": "model_to_export", "value": MODEL_TO_EXPORT},
    {"setting": "artifact_path", "value": str(MODEL_EXPORT_PATH)},
])
export_controls

### 5.2 Exportable predictor

The predictor stores the fitted estimator, training-data feature references, selected feature lists, and the two notebook functions that convert raw rows into model-ready inputs.

In [ ]:
class GoodreadsRatingPredictor:
    """Self-contained raw-input predictor exported from this notebook."""

    def __init__(
        self,
        fitted_record,
        feature_reference,
        metadata,
        feature_engineer=apply_feature_engineering,
        matrix_preparer=prepare_feature_matrix,
    ):
        self.model = fitted_record["model"]
        self.feature_reference = feature_reference
        self.numeric_features = list(fitted_record["numeric_features"])
        self.categorical_features = list(fitted_record["categorical_features"])
        self.metadata = metadata
        self._feature_engineer = feature_engineer
        self._matrix_preparer = matrix_preparer

    def prepare_features(self, raw_data: pd.DataFrame) -> pd.DataFrame:
        """Transform raw book rows into the columns expected by the fitted model."""
        engineered = self._feature_engineer(raw_data, self.feature_reference)
        return self._matrix_preparer(
            engineered,
            self.numeric_features,
            self.categorical_features,
        )

    def predict(self, raw_data: pd.DataFrame) -> np.ndarray:
        """Predict ratings from raw rows and keep predictions on the 0-to-5 scale."""
        model_input = self.prepare_features(raw_data)
        return np.clip(self.model.predict(model_input), 0, 5)


### 5.3 Refit and export

In [ ]:
exported_predictor = None

if not EXPORT_MODEL:
    print("Model export is disabled. Set EXPORT_MODEL = True to refit and export.")
else:
    matching_configs = [
        config for config in active_model_configs
        if config["model"] == MODEL_TO_EXPORT
    ]
    if len(matching_configs) != 1:
        raise ValueError(
            f"Expected exactly one active configuration for {MODEL_TO_EXPORT!r}; "
            f"found {len(matching_configs)}."
        )

    if MODEL_EXPORT_PATH.exists() and not OVERWRITE_MODEL:
        raise FileExistsError(
            f"Artifact already exists: {MODEL_EXPORT_PATH}. "
            "Set OVERWRITE_MODEL = True to replace it."
        )

    export_config = matching_configs[0]
    export_feature_reference = fit_feature_engineering_reference(export_X_raw, export_y)
    export_engineered_data = apply_feature_engineering(
        export_X_raw,
        export_feature_reference,
    )
    export_fitted_record = fit_model_configuration(
        export_config,
        export_engineered_data,
        export_y,
    )

    export_metadata = {
        "artifact_format_version": 1,
        "exported_at_utc": datetime.now(timezone.utc).isoformat(),
        "model_name": MODEL_TO_EXPORT,
        "training_dataset": EXPORT_TRAINING_DATASET,
        "parameters": export_fitted_record["parameters"],
        "training_rows": len(export_X_raw),
        "target": TARGET,
        "raw_input_columns": list(RAW_MODEL_INPUT_COLUMNS),
        "numeric_features": export_fitted_record["numeric_features"],
        "categorical_features": export_fitted_record["categorical_features"],
        "minimum_ratings_count_for_training": MIN_RATINGS_COUNT_FOR_MODELLING,
        "reference_year": REFERENCE_YEAR,
        "library_versions": {
            "python": platform.python_version(),
            "pandas": pd.__version__,
            "numpy": np.__version__,
            "scikit_learn": sklearn_package.__version__,
            "catboost": catboost_package.__version__,
            "xgboost": xgboost_package.__version__,
            "cloudpickle": cloudpickle.__version__,
        },
    }

    exported_predictor = GoodreadsRatingPredictor(
        fitted_record=export_fitted_record,
        feature_reference=export_feature_reference,
        metadata=export_metadata,
    )

    MODEL_EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with MODEL_EXPORT_PATH.open("wb") as artifact_file:
        cloudpickle.dump(exported_predictor, artifact_file)

    print(f"Exported {MODEL_TO_EXPORT} to {MODEL_EXPORT_PATH}")


### 5.4 Reload verification

Reloading the saved predictor and comparing its output on ordinary rows plus one deliberately imperfect row containing missing, invalid, unseen, and extra values.

In [ ]:
export_verification_summary = pd.DataFrame()

if not EXPORT_MODEL:
    print("Reload verification is skipped because export is disabled.")
else:
    with MODEL_EXPORT_PATH.open("rb") as artifact_file:
        reloaded_predictor = cloudpickle.load(artifact_file)

    verification_input = export_X_raw.head(10).copy().reset_index(drop=True)
    imperfect_row = verification_input.head(1).astype("object").copy()
    imperfect_row.loc[0, "authors"] = "Previously Unseen Author"
    imperfect_row.loc[0, "publisher"] = None
    imperfect_row.loc[0, "language_code"] = "new_language"
    imperfect_row.loc[0, "num_pages"] = np.nan
    imperfect_row.loc[0, "ratings_count"] = "250"
    imperfect_row.loc[0, "text_reviews_count"] = "not_available"
    imperfect_row.loc[0, "publication_date"] = "invalid_date"
    imperfect_row["extra_unused_column"] = "ignored safely"
    verification_input = pd.concat([verification_input, imperfect_row], ignore_index=True)

    predictions_before_reload = exported_predictor.predict(verification_input)
    predictions_after_reload = reloaded_predictor.predict(verification_input)
    np.testing.assert_allclose(predictions_before_reload, predictions_after_reload)
    assert reloaded_predictor.metadata == exported_predictor.metadata

    export_verification_summary = pd.DataFrame([{
        "artifact": str(MODEL_EXPORT_PATH),
        "training_dataset": EXPORT_TRAINING_DATASET,
        "file_size_bytes": MODEL_EXPORT_PATH.stat().st_size,
        "verified_rows": len(verification_input),
        "predictions_identical": True,
    }])


#### Show the verification summary

In [ ]:
export_verification_summary